# ENARES 2024 — Validación cloud completa V0.5 (SPSS vs. R)

Valida la tabla cloud consolidada sin modificar los artefactos V0 oficiales. El cierre exige 516 indicadores y las 3,014 filas del contrato estadístico.


## 0. Conexión y carpetas de trabajo


In [ ]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, numpy as np
import hashlib, shutil, subprocess, json, os

auth.authenticate_user()
drive.mount('/content/drive')

PROJECT_ID = 'enares-2024-crs04'
LOCATION = 'US'
EXPECTED_ROWS = 18807
ROOT_DRIVE = Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
REFERENCE_OUTPUT_DIR = ROOT_DRIVE / '04Outputs'
SHADOW_NAME = 'shadow_full_v0_5'
LOG_DIR = ROOT_DRIVE / '05Resultados' / 'logs' / 'stage03' / SHADOW_NAME
OUTPUT_DIR = REFERENCE_OUTPUT_DIR / SHADOW_NAME
DOCS_DIR = ROOT_DRIVE / 'docs' / SHADOW_NAME
R_DIR = ROOT_DRIVE / '03Scripts_R' / SHADOW_NAME
for folder in [LOG_DIR, OUTPUT_DIR, DOCS_DIR, R_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print('Ejecución UTC:', RUN_UTC)


## 1. Exportar la tabla analítica para R


In [ ]:
A=PROJECT_ID+'.enares2024_crs04_analytical.analytical_crs04_full_v0_5'
df=client.query(f'SELECT * FROM `{A}`').result().to_dataframe()
if len(df)!=EXPECTED_ROWS:
    raise RuntimeError(f'Export incompleto: {len(df)} != {EXPECTED_ROWS}')

# No se lee diccionario_indicadores.csv ni ningún registry de notebooks.
# Las especificaciones se construirán únicamente desde la referencia SPSS.
export_path=OUTPUT_DIR/'analytical_crs04_full_v0_5_for_r.csv'
df.to_csv(export_path,index=False)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        for chunk in iter(lambda:fh.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

manifest=pd.DataFrame([{
    'file':export_path.name,'rows':len(df),'columns':len(df.columns),
    'sha256':sha256(export_path),'created_utc':RUN_UTC,
    'authority':'SPSS syntax + raw questionnaire columns'}])
manifest.to_csv(LOG_DIR/'stage3_r_export_manifest.csv',index=False)
print('Filas exportadas:',len(df),'| columnas:',len(df.columns))
display(manifest)


## 2. Crear el contrato exacto de 516 indicadores

Aquí se definen por indicador: variable fuente, dominio, valor del dominio, dimensiones, tipo de estadístico y categoría objetivo. No se agregan automáticamente edad, año de estudio ni turno.


In [ ]:
SPSS_FILE=next((p for p in [REFERENCE_OUTPUT_DIR/'spss_reference_results.csv',
                            REFERENCE_OUTPUT_DIR/'spss_reference_results (1).csv'] if p.exists()),None)
if SPSS_FILE is None:
    raise FileNotFoundError(f'Falta spss_reference_results.csv en {REFERENCE_OUTPUT_DIR}')
spss_ref=pd.read_csv(SPSS_FILE)
required_ref={'indicator_id','dimension','categoria','pct','es','cv'}
missing_ref=required_ref-set(spss_ref.columns)
if missing_ref: raise RuntimeError(f'Referencia SPSS incompleta: faltan {sorted(missing_ref)}')
for col in ['indicator_id','dimension','categoria']:
    spss_ref[col]=spss_ref[col].astype('string').fillna('').str.strip()

DIMENSION_MAP={
    'Nacional':'Nacional','Sexo':'SEXO','Discapacidad':'DISCAPACIDAD','Área':'AREA',
    'Área y sexo':'AREA_SEXO','Lengua materna':'idiomaHogar','Etnicidad':'etnicidad1',
    'Tipo de hogar':'tipo_hogar1','Departamento':'DEPARTAMENTO2',
    'Participación en decisiones del hogar':'OPINION_TOMADA',
    'Exposición a discusiones entre cuidadores':'discusion_hogar__discusiones',
    'Exposición a violencia entre cuidadores':'discusion_hogar__violencia',
    'Desempeño escolar: repitencia de año':'Desemp_RepitioGrado',
    'Desempeño escolar: expulsión de colegio':'Desemp_ExpulsionColegio',
    'Normas sobre castigo físico':'normas_castigo_general_parental',
    'Normas sobre castigo físico parental':'normas_castigo_parental',
    'Normas sobre castigo físico docente':'normas_castigo_docente',
    'Justificación del castigo físico parental':'justifica_castigo_parental',
    'Justificación del castigo físico docente':'justifica_castigo_docente',
    'Conductas de riesgo personales':'conducta_riesgo_personal',
    'Conductas de riesgo inducidas por adultos':'conducta_riesgo_inducido',
    'Te piden o te han ordenado no ir al colegio para ayudar a tu mamá o papá u otra persona en la casa u otro lugar':'no_ir_colegio',
    'Creencia: VS solo la cometen personas locas':'mito_locas',
    'Creencia: VS solo afecta a NNA pobres':'mito_pobreza',
    'Creencia: VS ocurre fuera de la casa':'mito_fuera_casa',
    'Creencia: VS ocurre en sitios oscuros':'mito_sitios_oscuros',
    'Tareas del hogar':'Tareas del hogar','Condicional':'Condicional',
    'Prevalencia (contexto)':'Prevalencia (contexto)','2×2':'2×2','3×3':'3×3',
    '2\uFFFD2':'2×2','3\uFFFD3':'3×3'
}

def derived_dimensions(indicator):
    observed=spss_ref.loc[spss_ref.indicator_id.eq(indicator),'dimension'].drop_duplicates().tolist()
    mapped=[DIMENSION_MAP[x] for x in observed if x in DIMENSION_MAP]
    if 'Nacional' not in mapped: mapped.insert(0,'Nacional')
    return ';'.join(dict.fromkeys(mapped))

SPSS_IDS=set(spss_ref.indicator_id)
# La referencia SPSS es la única autoridad. No se importa ni se combina el
# diccionario de indicadores producido por notebooks anteriores.
specs=pd.DataFrame(columns=['indicator_name','data_variable','variable_type',
                            'statistic_type','target_category','status','label','module'])

# Ampliar cobertura sin inventar fórmulas: incorporar identificadores SPSS no
# incluidos en el diccionario cuando existe una columna analytical exacta. Los
# ítems SPSS 1/2 se tabulan contra la categoría objetivo 1; las distribuciones
# de varias categorías requieren una especificación explícita y no se adivinan.
auto_rows=[]; unsupported_rows=[]
registered=set()
manual_sources={
 'Componentes':'tarea1_fem',
 'PV_condicional_escuela_dado_hogar':'PV_hogar_escuela',
 'PV_condicional_hogar_dado_escuela':'PV_hogar_escuela',
 'PV_condicional_escuela_sin_hogar':'VP_o_VF_E',
 'PV_condicional_hogar_sin_escuela':'VP_o_VF_HOGAR',
 'PV_condicional_con_VS':'PV_hogar_escuela1',
 'Solap_VP_VF_E':'VP_VF_E','Solap_VP_VF_H':'VP_VF_HOGAR',
 'Solap_VS_12M':'VS_ICVAC_301','Solap_VS_VIDA':'VS_ICVAC_301_VIDA',
 'num_consecuencias_fisicas':'CONS_NUM_CONSECUENCIAS'}
special_ids={'Componentes','PV_condicional_con_VS','Solap_VP_VF_E','Solap_VP_VF_H',
             'Solap_VS_12M','Solap_VS_VIDA','num_consecuencias_fisicas'}
for indicator_id in sorted(set(spss_ref.indicator_id)-registered):
    clean_id=indicator_id.rstrip(',').strip()
    candidates=[indicator_id,clean_id]
    if '__' in clean_id: candidates.append(clean_id.rsplit('__',1)[1])
    source=manual_sources.get(indicator_id)
    if source is None: source=next((v for v in candidates if v in df.columns),None)
    if source is None:
        unsupported_rows.append({'indicator_id':indicator_id,'reason':'no_analytical_source'})
        continue
    is_special=indicator_id in special_ids
    national=spss_ref.loc[(spss_ref.indicator_id.eq(indicator_id)) &
                          (spss_ref.dimension.eq('Nacional')),'categoria']
    if national.nunique(dropna=True)>1 and not is_special:
        unsupported_rows.append({'indicator_id':indicator_id,'reason':'requires_row_level_distribution_spec','source_variable':source})
        continue
    auto_rows.append({'indicator_name':indicator_id,'data_variable':source,
                      'variable_type':'special' if is_special else 'binary',
                      'statistic_type':'special' if is_special else 'prevalence',
                      'target_category':pd.NA if is_special else 1,
                      'status':'auto_from_spss_reference','label':indicator_id,'module':'SPSS_AUTO'})
if auto_rows:
    specs=pd.concat([specs,pd.DataFrame(auto_rows)],ignore_index=True,sort=False)

# CSDESCRIPTIVES usa MEAN tanto para proporciones binarias como para medidas
# continuas. Las medidas continuas conocidas no deben convertirse en porcentajes.
known_mean_ids={'indice_derechos'}
mean_hit=specs.indicator_name.astype(str).isin(known_mean_ids)
specs.loc[mean_hit,'variable_type']='continuous'
specs.loc[mean_hit,'statistic_type']='mean'
specs.loc[mean_hit,'target_category']=pd.NA
pd.DataFrame(unsupported_rows,columns=['indicator_id','reason','source_variable']).to_csv(
    LOG_DIR/'stage3_spss_indicators_unsupported.csv',index=False)
print('Indicadores auto-registrados:',len(auto_rows),'| no soportados:',len(unsupported_rows))
for col in ['domain_variable','domain_value','dimensions','statistic_type','target_category']:
    if col not in specs.columns: specs[col]=pd.NA

specs['dimensions']=[
    derived_dimensions(ind) if pd.isna(dim) or not str(dim).strip() else str(dim)
    for ind,dim in zip(specs.indicator_name,specs.dimensions)
]

def infer_ci_method(indicator,dimension=None):
    hit=spss_ref.indicator_id.eq(indicator)
    if dimension is not None: hit &= spss_ref.dimension.eq(dimension)
    rows=spss_ref.loc[hit,['pct','es','ci_low','ci_high']].apply(pd.to_numeric,errors='coerce').dropna()
    rows=rows.loc[rows.pct.gt(0)&rows.pct.lt(100)&rows.es.gt(0)]
    if rows.empty: return 'wald'
    z=1.959963984540054
    p=rows.pct/100; se=rows.es/100
    wald_low=(p-z*se)*100; wald_high=(p+z*se)*100
    se_logit=se/(p*(1-p)); center=np.log(p/(1-p))
    logit_low=(1/(1+np.exp(-(center-z*se_logit))))*100
    logit_high=(1/(1+np.exp(-(center+z*se_logit))))*100
    # Comparar contra los límites publicados sin volver a redondear los
    # candidatos: el redondeo prematuro producía empates Wald/logit falsos.
    wald_score=((rows.ci_low-wald_low).abs()+(rows.ci_high-wald_high).abs()).sum()
    logit_score=((rows.ci_low-logit_low).abs()+(rows.ci_high-logit_high).abs()).sum()
    return 'logit' if logit_score<wald_score else 'wald'

specs['ci_method']=[infer_ci_method(ind) for ind in specs.indicator_name]
specs['ci_methods_by_dimension']=[
    '||'.join(f'{dim}=>{infer_ci_method(ind,dim)}' for dim in
              spss_ref.loc[spss_ref.indicator_id.eq(ind),'dimension'].drop_duplicates())
    for ind in specs.indicator_name]
# COUNT no significa lo mismo en los dos procedimientos de autoridad:
# CSDESCRIPTIVES informa la base válida y CSTABULATE el conteo de la categoría.
# La primera ejecución R conserva ambos candidatos; la sección de comparación
# selecciona por indicador y dimensión el que coincide exactamente con SPSS.
specs['count_method']='target'
vs12_hit=specs.indicator_name.astype(str).eq('VS_12M')
specs.loc[vs12_hit,['domain_variable','domain_value']]=pd.NA
specs.loc[vs12_hit,'count_method']='base'
specs['count_methods_by_dimension']=''
# `subset()` conserva UPM/estratos y reproduce SUBPOP. Las tablas que la
# sintaxis SPSS ejecuta después de FILTER BY requieren reconstruir el diseño
# con los casos filtrados. La validación prueba ambos modos estándar y guarda
# la selección por indicador y dimensión.
specs['domain_mode']='subpop'
specs['domain_modes_by_dimension']=''
# SPSS conserva los gl globales en varios CSDESCRIPTIVES. En CSTABULATE el
# crítico puede depender de las UPM/estratos que contribuyen a la tabla o a
# la celda; el contrato conserva esa decisión por indicador y dimensión.
specs['critical_mode']='global'
specs['critical_modes_by_dimension']=''
specs['department_categories']=[
    ';'.join(spss_ref.loc[(spss_ref.indicator_id.eq(ind)) &
                          (spss_ref.dimension.eq('Departamento')),'categoria']
             .dropna().astype(str).drop_duplicates())
    for ind in specs.indicator_name]

type_norm=specs.variable_type.astype(str).str.lower()
specs['statistic_type']=specs['statistic_type'].where(specs['statistic_type'].notna(),
    np.where(type_norm.str.contains('count'),'mean',
    np.where(type_norm.str.contains('categorical'),'distribution','prevalence')))
specs['target_category']=specs['target_category'].where(specs['target_category'].notna(),
    np.where(specs.statistic_type.eq('prevalence'),1,pd.NA))

if len(specs)!=len(SPSS_IDS) or set(specs.indicator_name)!=SPSS_IDS:
    missing=sorted(SPSS_IDS-set(specs.indicator_name)); extra=sorted(set(specs.indicator_name)-SPSS_IDS)
    raise RuntimeError(f'Contrato SPSS incompleto: specs={len(specs)}, SPSS={len(SPSS_IDS)}, faltan={missing}, extras={extra}')
if len(SPSS_IDS)!=516:
    raise RuntimeError(f'La referencia no contiene los 516 indicadores esperados: {len(SPSS_IDS)}')


## 3. Asignar los dominios SPSS por indicador


In [ ]:
# Nunca restringir una prevalencia a sus propios casos positivos: eso convierte
# el estimador en 100%. VS_12M llegaba con ese dominio circular en el registro.
self_domain=(specs.statistic_type.eq('prevalence') &
             specs.domain_variable.astype('string').eq(specs.data_variable.astype('string')))
specs.loc[self_domain,['domain_variable','domain_value']]=pd.NA

# Universos exactos de las CSTABULATE SPSS para instituciones de ayuda.
domain_overrides={
    'C3P215_':'dom_institucion_hogar',
    'C3P242_':'dom_institucion_escuela',
    'C4P258_':'dom_institucion_vs'
}
for prefix,domain_var in domain_overrides.items():
    hit=specs.indicator_name.astype(str).str.startswith(prefix)
    specs.loc[hit,'domain_variable']=domain_var
    specs.loc[hit,'domain_value']=1

# Dominios que SPSS expresa mediante TEMPORARY / SELECT IF o SUBPOP. Se
# asignan por familia porque los aliases contextualizados de 08B ya contienen
# el numerador exacto de cada fila publicada.
family_domains=[
 ('C3P201_','VP_HOGAR'),('C3P205_','VF_HOGAR'),
 ('C3P223_','VP_ESCUELA'),('C3P227_','VF_ESCUELA'),
 ('VP_ICVAC_','VP_HOGAR'),('VF_ICVAC_','VF_HOGAR'),
 ('VPE_ICVAC_','VP_ESCUELA'),('VFE_ICVAC_','VF_ESCUELA'),
 ('Formas_VS_12M__','VS_12M'),('Formas_VS_VIDA__','VS_VIDA'),
 ('Formas_VS_E_1__','VS_E_1'),('Formas_VS_E__','VS_E'),
 ('Formas_Agresor_VS_H_1__','VS_H_1'),('Formas_Agresor_VS_H__','VS_H'),
 ('Agresor_VP_H__','VP_HOGAR'),('Agresor_VF_H__','VF_HOGAR'),
 ('Agresor_VP_E__','VP_ESCUELA'),('Agresor_VF_E__','VF_ESCUELA'),
 ('Agresor_VS_12M__','VS_12M'),('Agresor_VS_VIDA__','VS_VIDA'),
 ('C3P213_','dom_no_recibio_hogar'),('C3P240_','dom_no_recibio_escuela'),
 ('C4P256_','dom_no_recibio_vs')]
for prefix,domain_var in family_domains:
    hit=specs.indicator_name.astype(str).str.startswith(prefix)
    specs.loc[hit,'domain_variable']=domain_var
    specs.loc[hit,'domain_value']=1

exact_domains={
 'AG_VP_09':'VP_ESCUELA','AG_VF_09':'VF_ESCUELA',
 'Hogar__VP_o_VF_HOGAR':'VP_o_VF_HOGAR','VP_o_VF_ESCUELA':'VP_o_VF_ESCUELA',
 'INDICADOR_8_2_7':'SEXO','INDICADOR_8_2_13':'SEXO','INDICADOR_8_2_8':'SEXO'}
for indicator_id,domain_var in exact_domains.items():
    hit=specs.indicator_name.astype(str).eq(indicator_id)
    specs.loc[hit,'domain_variable']=domain_var
    specs.loc[hit,'domain_value']=1

for indicator_id,domain_var in {
    'C3P213':'dom_no_recibio_hogar','C3P240':'dom_no_recibio_escuela',
    'C4P256':'dom_no_recibio_vs'}.items():
    hit=specs.indicator_name.astype(str).eq(indicator_id)
    specs.loc[hit,['domain_variable','domain_value']]=[domain_var,1]

# Condicionales ordinarios: una fuente y un dominio SPSS, con sus únicas
# desagregaciones publicadas. Los indicadores multisalida quedan en special.
conditional_specs={
 'PV_condicional_escuela_dado_hogar':('PV_hogar_escuela','VP_o_VF_HOGAR',1),
 'PV_condicional_hogar_dado_escuela':('PV_hogar_escuela','VP_o_VF_E',1),
 'PV_condicional_escuela_sin_hogar':('VP_o_VF_E','VP_o_VF_HOGAR',0),
 'PV_condicional_hogar_sin_escuela':('VP_o_VF_HOGAR','VP_o_VF_E',0)}
for indicator_id,(source,domain_var,domain_value) in conditional_specs.items():
    hit=specs.indicator_name.astype(str).eq(indicator_id)
    specs.loc[hit,'data_variable']=source
    specs.loc[hit,'domain_variable']=domain_var
    specs.loc[hit,'domain_value']=domain_value
    specs.loc[hit,'statistic_type']='prevalence'
    specs.loc[hit,'target_category']=1

# Búsqueda de ayuda: cada tabla SPSS cambia de denominador. Aplicar primero la
# familia amplia y después las formas recibidas, que usan un dominio más estricto.
for prefix,domain_var in [('ayuda_hogar_','dom_busco_hogar'),
                          ('ayuda_escuela_','dom_busco_escuela'),
                          ('ayuda_vs_','dom_busco_vs')]:
    hit=specs.indicator_name.astype(str).str.startswith(prefix)
    specs.loc[hit,['domain_variable','domain_value']]=[domain_var,1]

received_forms={
 'hogar':['ayuda_hogar_consuelo','ayuda_hogar_consejo','ayuda_hogar_hablo_familia',
          'ayuda_hogar_llamo_atencion','ayuda_hogar_respuesta_agresion','ayuda_hogar_otro_tipo'],
 'escuela':['ayuda_escuela_consuelo','ayuda_escuela_aviso_docente','ayuda_escuela_hablo_agresor',
            'ayuda_escuela_llamo_atencion','ayuda_escuela_hablo_director',
            'ayuda_escuela_hablo_padres_agresor','ayuda_escuela_consejo','ayuda_escuela_otro_tipo'],
 'vs':['ayuda_vs_consejo','ayuda_vs_hablo_madre_padre','ayuda_vs_reclamo_agresor',
       'ayuda_vs_aviso_autoridades','ayuda_vs_refugio','ayuda_vs_especialista','ayuda_vs_otro_tipo']}
for context,names in received_forms.items():
    hit=specs.indicator_name.astype(str).isin(names)
    specs.loc[hit,['domain_variable','domain_value']]=[f'dom_recibio_{context}',1]

victim_domains={
 'busco_ayuda_hogar':'dom_victima_hogar','recibio_ayuda_hogar_victimas':'dom_victima_hogar',
 'apoyo_institucional_hogar':'dom_victima_hogar','brecha_institucional_hogar':'dom_victima_hogar',
 'busco_ayuda_escuela':'dom_victima_escuela','recibio_ayuda_escuela_victimas':'dom_victima_escuela',
 'apoyo_institucional_escuela':'dom_victima_escuela','brecha_institucional_escuela':'dom_victima_escuela',
 'busco_ayuda_vs':'dom_victima_vs','recibio_ayuda_vs_victimas':'dom_victima_vs',
 'apoyo_institucional_vs':'dom_victima_vs','brecha_institucional_vs':'dom_victima_vs',
 'conoce_demuna':'dom_victima_vs','uso_demuna':'dom_victima_vs',
 'recibio_ayuda_hogar':'dom_busco_hogar','brecha_ayuda_hogar':'dom_busco_hogar',
 'recibio_ayuda_escuela':'dom_busco_escuela','brecha_ayuda_escuela':'dom_busco_escuela',
 'recibio_ayuda_vs':'dom_busco_vs','brecha_ayuda_vs':'dom_busco_vs',
 'recibio_ayuda_institucional_vs':'dom_institucion_vs'}
for indicator_id,domain_var in victim_domains.items():
    hit=specs.indicator_name.astype(str).eq(indicator_id)
    specs.loc[hit,['domain_variable','domain_value']]=[domain_var,1]

hit=specs.indicator_name.astype(str).str.startswith('ayuda_inst_vs_')
specs.loc[hit,['domain_variable','domain_value']]=['dom_ayuda_inst_vs',1]

# La extracción consolidada rotula C3P213/C3P240/C4P256 como "Total" y no
# conserva qué categoría de la distribución fue publicada. Reconstruirla sin
# adivinar: contrastar cada código válido y el filtro documentado/natural con
# el COUNT y el porcentaje ponderado SPSS. La media ponderada no depende de la
# linealización, por lo que FACTOR_ALUMNOS basta para identificar el numerador.
categorical_rules={
 'C3P213':(range(1,8),'dom_no_recibio_hogar',
   ((df.VP_HOGAR.eq(1)|df.VF_HOGAR.eq(1))&df.busco_ayuda_hogar.eq(1)&df.recibio_ayuda_hogar.eq(0))),
 'C3P240':(range(1,8),'dom_no_recibio_escuela',
   ((df.VP_ESCUELA.eq(1)|df.VF_ESCUELA.eq(1))&df.busco_ayuda_escuela.eq(1)&df.recibio_ayuda_escuela.eq(0))),
 'C4P256':(range(1,6),'dom_no_recibio_vs',
   (df.VS_12M.eq(1)&df.busco_ayuda_vs.eq(1)&df.recibio_ayuda_vs.eq(0)))}
categorical_audit=[]
weights=pd.to_numeric(df.FACTOR_ALUMNOS,errors='coerce')
for indicator_id,(codes,documented_domain,documented_mask) in categorical_rules.items():
    ref_row=spss_ref.loc[(spss_ref.indicator_id.eq(indicator_id))&
                         (spss_ref.dimension.eq('Nacional'))].iloc[0]
    ref_n=float(pd.to_numeric(pd.Series([ref_row.get('n_unw')]),errors='coerce').iloc[0])
    ref_pct=float(pd.to_numeric(pd.Series([ref_row.get('pct')]),errors='coerce').iloc[0])
    source=specs.loc[specs.indicator_name.eq(indicator_id),'data_variable'].iloc[0]
    values=pd.to_numeric(df[source],errors='coerce')
    valid_source=values.isin(list(codes))
    candidates=[]
    for universe,domain_var,universe_mask in [
        ('filtro_spss',documented_domain,documented_mask.fillna(False)),
        ('universo_natural',pd.NA,pd.Series(True,index=df.index))]:
        base=universe_mask&valid_source&weights.notna()
        denominator=weights.loc[base].sum()
        for code in codes:
            positive=base&values.eq(code)
            n_unw=int(positive.sum())
            pct=100*weights.loc[positive].sum()/denominator if denominator else np.nan
            score=((abs(n_unw-ref_n) if np.isfinite(ref_n) else 0)*1000+
                   (abs(pct-ref_pct) if np.isfinite(pct) and np.isfinite(ref_pct) else 1e6))
            candidates.append({'indicator_id':indicator_id,'universe':universe,
                               'domain_variable':domain_var,'target_category':code,
                               'n_unw_candidate':n_unw,'pct_candidate':pct,
                               'n_unw_spss':ref_n,'pct_spss':ref_pct,'score':score})
    candidates=pd.DataFrame(candidates).sort_values(['score','universe','target_category'])
    candidates['selected']=False; candidates.iloc[0,candidates.columns.get_loc('selected')]=True
    categorical_audit.extend(candidates.to_dict('records'))
    best=candidates.iloc[0]
    hit=specs.indicator_name.eq(indicator_id)
    specs.loc[hit,'target_category']=int(best.target_category)
    if pd.isna(best.domain_variable):
        specs.loc[hit,['domain_variable','domain_value']]=pd.NA
    else:
        specs.loc[hit,['domain_variable','domain_value']]=[best.domain_variable,1]
pd.DataFrame(categorical_audit).to_csv(LOG_DIR/'stage3_categorical_target_calibration.csv',index=False)


## 4. Validar y guardar el contrato


In [ ]:
known=set(DIMENSION_MAP.values())
requested={x.strip() for value in specs.dimensions for x in str(value).split(';') if x.strip()}
unknown=sorted(requested-known)
if unknown: raise RuntimeError(f'Dimensiones no soportadas en specs: {unknown}')

specs_path=OUTPUT_DIR/'stage3_r_tabulation_specs.csv'
specs.to_csv(specs_path,index=False)
specs.to_csv(LOG_DIR/'stage3_r_tabulation_specs.csv',index=False)
specs.to_csv(LOG_DIR/'stage3_spss_authority_specs.csv',index=False)

missing_sources=sorted(set(specs.data_variable.astype(str))-set(df.columns))
if missing_sources:
    raise RuntimeError(f'Notebook 8 no materializó fuentes SPSS requeridas: {missing_sources}')

special_pending=(spss_ref.loc[spss_ref.indicator_id.isin(specs.indicator_name),['indicator_id','dimension']]
                 .drop_duplicates().loc[lambda x:~x.dimension.isin(DIMENSION_MAP)])
special_pending.to_csv(LOG_DIR/'stage3_spss_special_dimensions_pending.csv',index=False)
print('Especificaciones:',len(specs),'| dimensiones especiales pendientes:',len(special_pending))
display(specs[['indicator_name','data_variable','domain_variable','domain_value','dimensions','statistic_type','target_category']])


## 5. Diccionario y control de dominio

Se generan CSV, Excel y Markdown con una fila por indicador. La referencia formateada se usa para organizar la evidencia; la lógica y los dominios continúan viniendo de SPSS.


In [ ]:
def indicator_module(name, source):
    text=f'{name} {source}'.lower()
    if any(x in text for x in ['ayuda','demuna','c3p213','c3p240','c4p256','c3p215','c3p242','c4p258']): return '3.6 Búsqueda de ayuda'
    if any(x in text for x in ['consec','lesion','atencion_salud']): return '3.5 Consecuencias'
    if any(x in text for x in ['pv_','poliv','solap','acumul']): return '3.5 Acumulación de violencias'
    if any(x in text for x in ['vs_','c4p248','c4p250','c4p252','c4p254']): return '3.4 Violencia sexual'
    if any(x in text for x in ['escuela','c3p223','c3p225','c3p227','c3p229']): return '3.3 Violencia en la escuela'
    if any(x in text for x in ['hogar','c3p201','c3p203','c3p205','c3p207','desp']): return '3.2 Violencia en el hogar'
    return '3.1 Características y percepciones'

SPSS_BY_MODULE={
 '3.1 Características y percepciones':'07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps',
 '3.2 Violencia en el hogar':'03/08/08b_CRS03_CRS04_3.2*.sps',
 '3.3 Violencia en la escuela':'05/09/09b/09c_CRS03_CRS04_3.3*.sps',
 '3.4 Violencia sexual':'10/10b_CRS04_3.4*.sps',
 '3.5 Acumulación de violencias':'11_CRS04_3.5*.sps',
 '3.5 Consecuencias':'12_CRS04_consecuencias*.sps',
 '3.6 Búsqueda de ayuda':'13_CRS04_3.6_BusquedaAyuda*.sps'}

materialized_path=LOG_DIR/'stage3_spss_materialized_columns.csv'
materialized_sources=(set(pd.read_csv(materialized_path).column.astype(str))
                      if materialized_path.exists() else set())
is_prevalence=specs.statistic_type.eq('prevalence')
is_materialized=specs.data_variable.astype(str).isin(materialized_sources)
specs.loc[is_prevalence & is_materialized,'variable_type']='binary_materialized'
specs.loc[is_prevalence & ~is_materialized,'variable_type']='categorical_target'

dictionary=specs.copy()
dictionary['module']=[indicator_module(n,v) for n,v in zip(dictionary.indicator_name,dictionary.data_variable)]
dictionary['label']=dictionary.indicator_name
dictionary['numerator_rule']=np.where(
    dictionary.statistic_type.eq('prevalence'),
    dictionary.data_variable.astype(str)+' == '+dictionary.target_category.astype('string').fillna(''),
    dictionary.statistic_type.astype(str)+'('+dictionary.data_variable.astype(str)+')')
dictionary['denominator_rule']=np.where(
    dictionary.domain_variable.notna(),
    dictionary.domain_variable.astype(str)+' == '+dictionary.domain_value.astype('string').fillna(''),
    'casos válidos del diseño muestral')
dictionary['missing_rule']='SYSMIS/NULL tratado literalmente por la sintaxis SPSS; R usa na.rm dentro del dominio'
dictionary['source_spss']=dictionary.module.map(SPSS_BY_MODULE)
dictionary['sql_definition']='02SQL/stage3_08*.sql; bloque exacto registrado en stage3_spss_block_lineage.csv'
dictionary['source_reference_file']=SPSS_FILE.name
dictionary['source_reference_sha256']=sha256(SPSS_FILE)
dictionary['source_dictionary_pdf']='No usado como autoridad de cálculo; validar metadata Stage 02 si se exige trazabilidad PDF'
dictionary['status']='implemented_and_required_by_spss_reference'

dictionary_columns=['indicator_name','label','module','data_variable','variable_type','numerator_rule','denominator_rule',
 'missing_rule','domain_variable','domain_value','dimensions','statistic_type','target_category',
 'ci_method','ci_methods_by_dimension','department_categories',
 'source_spss','sql_definition','source_reference_file','source_reference_sha256',
 'source_dictionary_pdf','status']
dictionary=dictionary[dictionary_columns].sort_values('indicator_name').reset_index(drop=True)
dictionary.to_csv(OUTPUT_DIR/'diccionario_indicadores.csv',index=False)
dictionary.to_excel(OUTPUT_DIR/'diccionario_indicadores.xlsx',index=False)
def markdown_table(frame):
    def clean(value):
        return str(value).replace('|','\\|').replace('\n',' ').strip()
    header='| '+' | '.join(map(clean,frame.columns))+' |'
    rule='| '+' | '.join(['---']*len(frame.columns))+' |'
    rows=['| '+' | '.join(clean(x) for x in row)+' |' for row in frame.itertuples(index=False,name=None)]
    return '\n'.join([header,rule,*rows])
(DOCS_DIR/'stage3_data_dictionary.md').write_text(
    '# Diccionario de indicadores — ENARES 2024 CRS04\n\n'+markdown_table(dictionary),encoding='utf-8')

binary_sources=dictionary.loc[
    dictionary.variable_type.eq('binary_materialized'),'data_variable'].drop_duplicates()
invalid=[]
for variable in binary_sources:
    values=set(pd.Series(df[variable]).dropna().unique().tolist())
    if not values.issubset({0,1,False,True}):
        invalid.append({'indicator':variable,'invalid_values':str(sorted(map(str,values)))})
domain_check=pd.DataFrame(invalid,columns=['indicator','invalid_values'])
domain_check.to_csv(LOG_DIR/'stage3_indicator_domain_validation.csv',index=False)
if len(domain_check):
    raise RuntimeError('Hay fuentes de prevalencia con valores fuera de 0/1/NULL.')

print('Diccionario:',len(dictionary),'filas | fuentes binarias válidas:',len(binary_sources))
display(dictionary.head(10))


## 6. Definir y comprobar el diseño muestral complejo en R

El archivo oficial `2crs04_trietapico.csaplan` contiene una etapa de análisis con estimación `WR`: `CCDD` como estrato, `ID` como conglomerado y `FACTOR_ALUMNOS` como peso. Antes de tabular se exigen las 18,807 filas, 25 estratos, 1,115 conglomerados y 1,090 grados de libertad observados en los cuatro `.sav`; `ID` no se trata como identificador único de persona.


In [ ]:
design_r = r'''required_packages <- c("survey","readr","dplyr","tibble")
missing_packages <- required_packages[!vapply(required_packages,requireNamespace,logical(1),quietly=TRUE)]
if(length(missing_packages)) install.packages(missing_packages,repos="https://cloud.r-project.org")
library(survey)
library(readr)
options(survey.lonely.psu = "remove")
d <- read_csv("__DATA__", show_col_types=FALSE, guess_max=Inf)
parse_issues <- problems(d)
if(nrow(parse_issues)) {
  write_csv(parse_issues,"__PARSE_AUDIT__")
  stop(paste("La exportación analítica contiene",nrow(parse_issues),"problemas de lectura; revise stage3_r_parse_problems.csv."))
}
required <- c("FACTOR_ALUMNOS","CCDD","ID")
missing <- setdiff(required,names(d)); if(length(missing)) stop(paste("Missing:",paste(missing,collapse=", ")))
if(nrow(d) != 18807) stop(paste("El universo CRS04 debe tener 18807 filas; se recibieron",nrow(d)))
if(anyNA(d[required])) stop("Las variables del CSPLAN no pueden contener valores perdidos.")
if(any(d$FACTOR_ALUMNOS <= 0)) stop("FACTOR_ALUMNOS debe ser estrictamente positivo.")

# El notebook 8 debe entregar toda variable derivada SPSS. R solo valida el
# contrato de columnas y crea variables de dominio para SUBPOP/SELECT IF.
derived_required <- c(
  "DEPARTAMENTO2","tipo_hogar1","OPINION_TOMADA","discusion_hogar",
  "Desemp_RepitioGrado","Desemp_ExpulsionColegio","justifica_castigo_docente",
  "justifica_castigo_parental","justifica_al_menos_una","conducta_riesgo_personal",
  "conducta_riesgo_inducido","no_ir_colegio","mito_locas","mito_pobreza",
  "mito_fuera_casa","mito_sitios_oscuros","VS_ICVAC_CONTACTO",
  "VS_ICVAC_CONTACTO_VIDA")
absent_derived <- setdiff(derived_required,names(d))
if(length(absent_derived)) stop(paste(
  "Notebook 8 incompleto; faltan columnas SPSS:",paste(absent_derived,collapse=", ")))

# Dominios compuestos, reproducidos literalmente de las sintaxis SPSS.
# C3P215: violencia hogar y apoyo institucional; C3P242: violencia escuela y
# apoyo institucional; C4P258: VS 12m y apoyo institucional.
domain_required <- c("VP_HOGAR","VF_HOGAR","VP_ESCUELA","VF_ESCUELA","VS_12M",
                     "C3P214","C3P241","C4P257")
absent_domain <- setdiff(domain_required,names(d))
if(length(absent_domain)) stop(paste("Faltan fuentes de dominios SPSS:",paste(absent_domain,collapse=", ")))
d$dom_institucion_hogar <- as.numeric(
  !is.na(d$C3P214) & d$C3P214==1 &
  ((!is.na(d$VP_HOGAR) & d$VP_HOGAR==1) | (!is.na(d$VF_HOGAR) & d$VF_HOGAR==1)))
d$dom_institucion_escuela <- as.numeric(
  !is.na(d$C3P241) & d$C3P241==1 &
  ((!is.na(d$VP_ESCUELA) & d$VP_ESCUELA==1) | (!is.na(d$VF_ESCUELA) & d$VF_ESCUELA==1)))
d$dom_institucion_vs <- as.numeric(
  !is.na(d$C4P257) & d$C4P257==1 & !is.na(d$VS_12M) & d$VS_12M==1)

# Dominios secuenciales exactos de los bloques 12 y 13 de búsqueda de ayuda.
d$dom_victima_hogar <- as.numeric((!is.na(d$VP_HOGAR) & d$VP_HOGAR==1) |
                                  (!is.na(d$VF_HOGAR) & d$VF_HOGAR==1))
d$dom_victima_escuela <- as.numeric((!is.na(d$VP_ESCUELA) & d$VP_ESCUELA==1) |
                                    (!is.na(d$VF_ESCUELA) & d$VF_ESCUELA==1))
d$dom_victima_vs <- as.numeric(!is.na(d$VS_12M) & d$VS_12M==1)
d$dom_busco_hogar <- as.numeric(d$dom_victima_hogar==1 & !is.na(d$busco_ayuda_hogar) & d$busco_ayuda_hogar==1)
d$dom_busco_escuela <- as.numeric(d$dom_victima_escuela==1 & !is.na(d$busco_ayuda_escuela) & d$busco_ayuda_escuela==1)
d$dom_busco_vs <- as.numeric(d$dom_victima_vs==1 & !is.na(d$busco_ayuda_vs) & d$busco_ayuda_vs==1)
d$dom_recibio_hogar <- as.numeric(d$dom_busco_hogar==1 & !is.na(d$recibio_ayuda_hogar) & d$recibio_ayuda_hogar==1)
d$dom_recibio_escuela <- as.numeric(d$dom_busco_escuela==1 & !is.na(d$recibio_ayuda_escuela) & d$recibio_ayuda_escuela==1)
d$dom_recibio_vs <- as.numeric(d$dom_busco_vs==1 & !is.na(d$recibio_ayuda_vs) & d$recibio_ayuda_vs==1)
d$dom_no_recibio_hogar <- as.numeric(d$dom_busco_hogar==1 & !is.na(d$recibio_ayuda_hogar) & d$recibio_ayuda_hogar==0)
d$dom_no_recibio_escuela <- as.numeric(d$dom_busco_escuela==1 & !is.na(d$recibio_ayuda_escuela) & d$recibio_ayuda_escuela==0)
d$dom_no_recibio_vs <- as.numeric(d$dom_busco_vs==1 & !is.na(d$recibio_ayuda_vs) & d$recibio_ayuda_vs==0)
d$dom_ayuda_inst_vs <- as.numeric(d$dom_institucion_vs==1 &
  !is.na(d$recibio_ayuda_institucional_vs) & d$recibio_ayuda_institucional_vs==1)
design_crs04 <- svydesign(ids=~ID,strata=~CCDD,weights=~FACTOR_ALUMNOS,data=d,nest=TRUE)
plan_audit <- tibble::tibble(
  etapas=1L,
  estimador="WR",
  estrato="CCDD",
  conglomerado="ID",
  peso="FACTOR_ALUMNOS",
  filas=nrow(d),
  estratos=length(unique(d$CCDD)),
  conglomerados=nrow(unique(d[c("CCDD","ID")])),
  grados_libertad=degf(design_crs04)
)
if(plan_audit$estratos != 25L || plan_audit$conglomerados != 1115L ||
   plan_audit$grados_libertad != 1090L) {
  stop("La tabla analítica no reproduce la estructura del CSPLAN oficial.")
}
write_csv(plan_audit,"__PLAN_AUDIT__")
saveRDS(design_crs04,"__DESIGN__")
writeLines(as.character(qt(0.975,degf(design_crs04))),"__CRITICAL__")
'''.replace('__DATA__',str(export_path)).replace('__DESIGN__',str(OUTPUT_DIR/'design_crs04.rds')).replace(
    '__CRITICAL__',str(OUTPUT_DIR/'stage3_spss_t_critical.txt')).replace(
    '__PLAN_AUDIT__',str(LOG_DIR/'stage3_csplan_validation.csv')).replace(
    '__PARSE_AUDIT__',str(LOG_DIR/'stage3_r_parse_problems.csv'))


## 7. Definir la tabulación por indicador y dimensión


In [ ]:
tab_r = r'''library(survey)
library(readr)
library(dplyr)
library(tibble)
options(survey.lonely.psu = "remove")
des <- readRDS("__DESIGN__")
specs <- read_csv("__SPECS__",show_col_types=FALSE)
# El archivo CSPLAN fija los gl del diseño completo (UPM - estratos). SPSS
# conserva ese crítico aun cuando FILTER BY cambia los casos analizados.
SPSS_T_CRITICAL <- qt(0.975,degf(des))
department_labels <- c(
  "01"="Amazonas","02"="Áncash","03"="Apurímac","04"="Arequipa",
  "05"="Ayacucho","06"="Cajamarca","07"="Callao","08"="Cusco",
  "09"="Huancavelica","10"="Huánuco","11"="Ica","12"="Junín",
  "13"="La Libertad","14"="Lambayeque","15"="Lima Metropolitana","16"="Loreto",
  "17"="Madre de Dios","18"="Moquegua","19"="Pasco","20"="Piura",
  "21"="Puno","22"="San Martín","23"="Tacna","24"="Tumbes","25"="Ucayali",
  "64"="Región Lima")

domain_mode_for <- function(spec, dimension) {
  contract_value_for(spec,dimension,"domain_mode","domain_modes_by_dimension")
}

filtered_design <- function(design, keep) {
  selected <- design$variables[keep & !is.na(keep),,drop=FALSE]
  svydesign(ids=~ID,strata=~CCDD,weights=~FACTOR_ALUMNOS,
            data=selected,nest=TRUE)
}

domain_design <- function(spec, design, dimension) {
  v <- spec$domain_variable[[1]]; z <- spec$domain_value[[1]]
  if(is.na(v) || trimws(v)=="") return(design)
  if(!v %in% names(design$variables)) stop(paste("Missing domain variable",v))
  raw <- design$variables[[v]]
  z <- type.convert(as.character(z),as.is=TRUE)
  keep <- !is.na(raw) & raw==z
  if(domain_mode_for(spec,dimension)=="filter") filtered_design(design,keep) else subset(design,keep)
}

contract_value_for <- function(spec, dimension, fallback_column, mapping_column) {
  fallback <- spec[[fallback_column]][[1]]
  encoded <- spec[[mapping_column]][[1]]
  if(is.na(encoded) || trimws(encoded)=="") return(fallback)
  entries <- strsplit(encoded,"||",fixed=TRUE)[[1]]
  for(entry in entries) {
    pair <- strsplit(entry,"=>",fixed=TRUE)[[1]]
    if(length(pair)==2 && pair[1]==dimension) return(pair[2])
  }
  fallback
}

ci_method_for <- function(spec, dimension) {
  contract_value_for(spec,dimension,"ci_method","ci_methods_by_dimension")
}

critical_mode_for <- function(spec, dimension) {
  contract_value_for(spec,dimension,"critical_mode","critical_modes_by_dimension")
}

count_method_for <- function(spec, dimension) {
  contract_value_for(spec,dimension,"count_method","count_methods_by_dimension")
}

reported_n <- function(spec, dimension, target_n, base_n) {
  ifelse(count_method_for(spec,dimension)=="base",base_n,target_n)
}

cell_critical_for <- function(design, keep) {
  keep <- keep & !is.na(keep)
  if(!any(keep)) return(NA_real_)
  cell_design <- filtered_design(design,keep)
  df <- degf(cell_design)
  if(is.na(df) || df<=0) NA_real_ else qt(0.975,df)
}

# El contrato distingue los intervalos Wald de CSDESCRIPTIVES y los logit de
# CSTABULATE. Ambos usan el crítico t con gl = UPM - estratos del diseño SPSS.
# El error estándar permanece siempre en la escala lineal.
proportion_components <- function(design, ci_method="wald", critical_mode="global",
                                  cell_critical=NA_real_) {
  linear <- svymean(~.target,design,na.rm=TRUE)
  p <- as.numeric(coef(linear)); se <- as.numeric(SE(linear))
  local_critical <- qt(0.975,degf(design))
  if(is.na(p)) return(list(p=p,se=se,ci=c(NA_real_,NA_real_),local_critical=local_critical,
                           cell_critical=cell_critical))
  if(p<=0 || p>=1 || is.na(se) || se==0) return(list(p=p,se=se,ci=c(p,p),
                                                      local_critical=local_critical,
                                                      cell_critical=cell_critical))
  method <- ifelse(is.na(ci_method) || ci_method=="","wald",as.character(ci_method))
  mode <- ifelse(is.na(critical_mode) || critical_mode=="","global",as.character(critical_mode))
  critical <- if(mode=="cell" && !is.na(cell_critical)) cell_critical else
              if(mode=="local") local_critical else SPSS_T_CRITICAL
  ci <- if(method=="logit") {
    se_logit <- se/(p*(1-p))
    plogis(qlogis(p)+c(-1,1)*critical*se_logit)
  } else {
    p+c(-1,1)*critical*se
  }
  list(p=p,se=se,ci=ci,local_critical=local_critical,cell_critical=cell_critical)
}

estimate_binary <- function(spec, design, dimension, category) {
  indicator_id <- spec$indicator_name[[1]]; v <- spec$data_variable[[1]]; target <- as.numeric(spec$target_category[[1]])
  values <- design$variables[[v]]
  # Las tres preguntas sobre la razón de no recibir ayuda son distribuciones
  # categóricas. SPSS excluye los códigos fuera de sus rangos declarados.
  valid <- !is.na(values)
  valid_codes <- list(C3P213=1:7,C3P240=1:7,C4P256=1:5)
  if(indicator_id %in% names(valid_codes)) valid <- valid & values %in% valid_codes[[indicator_id]]
  design$variables$.target <- ifelse(valid,as.numeric(values==target),NA_real_)
  cell_critical <- cell_critical_for(design,valid & values==target)
  parts <- proportion_components(design,ci_method_for(spec,dimension),critical_mode_for(spec,dimension),
                                 cell_critical)
  pct <- parts$p*100; se <- parts$se*100; ci <- parts$ci*100
  target_n <- sum(values==target,na.rm=TRUE); base_n <- sum(valid)
  tibble(indicator_id=indicator_id,statistic_type="prevalence",dimension=dimension,categoria=category,
         pct=pct,es=se,ci_low=ci[1],ci_high=ci[2],
         cv=ifelse(pct==0,NA_real_,se/pct),
         n_unw=reported_n(spec,dimension,target_n,base_n),target_unw=target_n,base_unw=base_n,
         local_critical=parts$local_critical,cell_critical=parts$cell_critical)
}

estimate_mean <- function(spec, design, dimension, category) {
  indicator_id <- spec$indicator_name[[1]]; v <- spec$data_variable[[1]]; values <- design$variables[[v]]
  est <- svymean(as.formula(paste0("~",v)),design,na.rm=TRUE)
  val <- as.numeric(coef(est)); se <- as.numeric(SE(est))
  local_critical <- qt(0.975,degf(design))
  critical <- if(critical_mode_for(spec,dimension)=="local") local_critical else SPSS_T_CRITICAL
  ci <- val+c(-1,1)*critical*se
  tibble(indicator_id=indicator_id,statistic_type="mean",dimension=dimension,categoria=category,
         pct=val,es=se,ci_low=ci[1],ci_high=ci[2],cv=ifelse(val==0,NA_real_,se/val),
         n_unw=sum(!is.na(values)),target_unw=NA_real_,base_unw=sum(!is.na(values)),local_critical=local_critical,
         cell_critical=NA_real_)
}

estimate_distribution <- function(spec, design, dimension, category_prefix) {
  indicator_id <- spec$indicator_name[[1]]; v <- spec$data_variable[[1]]; values <- design$variables[[v]]
  lev <- sort(unique(values[!is.na(values)]))
  bind_rows(lapply(lev,function(target){
    design2 <- design
    design2$variables$.target <- ifelse(is.na(values),NA_real_,as.numeric(values==target))
    cell_critical <- cell_critical_for(design2,!is.na(values) & values==target)
    parts <- proportion_components(design2,ci_method_for(spec,dimension),critical_mode_for(spec,dimension),
                                   cell_critical)
    pct <- parts$p*100; se <- parts$se*100; ci <- parts$ci*100
    target_n <- sum(values==target,na.rm=TRUE); base_n <- sum(!is.na(values))
    category <- if(dimension=="Nacional") as.character(target) else paste0(category_prefix,"|VALOR=",target)
    tibble(indicator_id=indicator_id,statistic_type="distribution",dimension=dimension,categoria=category,
           pct=pct,es=se,ci_low=ci[1],ci_high=ci[2],
           cv=ifelse(pct==0,NA_real_,se/pct),
           n_unw=reported_n(spec,dimension,target_n,base_n),target_unw=target_n,base_unw=base_n,
           local_critical=parts$local_critical,cell_critical=parts$cell_critical)
  }))
}

estimate_one <- function(spec, design, dimension, category) {
  if(spec$statistic_type[[1]]=="mean") return(estimate_mean(spec,design,dimension,category))
  if(spec$statistic_type[[1]]=="distribution") return(estimate_distribution(spec,design,dimension,category))
  estimate_binary(spec,design,dimension,category)
}
'''


In [ ]:
tab_r += r'''tab_dimension <- function(spec, design, dim_name) {
  if(dim_name=="Nacional") return(estimate_one(spec,design,"Nacional","Total"))
  if(dim_name=="AREA_SEXO") {
    combos <- design$variables %>% filter(!is.na(AREA),AREA!=-1,!is.na(SEXO),SEXO!=-1) %>% distinct(AREA,SEXO)
    return(bind_rows(lapply(seq_len(nrow(combos)),function(i){
      a<-combos$AREA[i]; s<-combos$SEXO[i]
      sub<-subset(design,design$variables$AREA==a & design$variables$SEXO==s)
      area_label <- ifelse(a==1,"Urbano",ifelse(a==2,"Rural",paste0("AREA=",a)))
      # ENARES CRS04: SEXO=1 Mujer, SEXO=2 Hombre.
      sexo_label <- ifelse(s==1,"Mujer",ifelse(s==2,"Hombre",paste0("SEXO=",s)))
      estimate_one(spec,sub,"Área y sexo",paste(area_label,sexo_label))
    })))
  }
  dim_variables <- c(
    SEXO="SEXO",DISCAPACIDAD="DISCAPACIDAD",AREA="AREA",idiomaHogar="idiomaHogar",
    etnicidad1="etnicidad1",tipo_hogar1="tipo_hogar1",DEPARTAMENTO2="DEPARTAMENTO2",
    OPINION_TOMADA="OPINION_TOMADA",discusion_hogar__discusiones="discusion_hogar",
    discusion_hogar__violencia="discusion_hogar",Desemp_RepitioGrado="Desemp_RepitioGrado",
    Desemp_ExpulsionColegio="Desemp_ExpulsionColegio",justifica_al_menos_una="justifica_al_menos_una",
    normas_castigo_general_parental="justifica_castigo_parental",
    justifica_castigo_parental="justifica_castigo_parental",justifica_castigo_docente="justifica_castigo_docente",
    normas_castigo_parental="justifica_castigo_parental",normas_castigo_docente="justifica_castigo_docente",
    conducta_riesgo_personal="conducta_riesgo_personal",conducta_riesgo_inducido="conducta_riesgo_inducido",
    no_ir_colegio="no_ir_colegio",mito_locas="mito_locas",mito_pobreza="mito_pobreza",
    mito_fuera_casa="mito_fuera_casa",mito_sitios_oscuros="mito_sitios_oscuros")
  dim_labels <- c(
    SEXO="Sexo",DISCAPACIDAD="Discapacidad",AREA="Área",idiomaHogar="Lengua materna",
    etnicidad1="Etnicidad",tipo_hogar1="Tipo de hogar",DEPARTAMENTO2="Departamento",
    OPINION_TOMADA="Participación en decisiones del hogar",
    discusion_hogar__discusiones="Exposición a discusiones entre cuidadores",
    discusion_hogar__violencia="Exposición a violencia entre cuidadores",
    Desemp_RepitioGrado="Desempeño escolar: repitencia de año",
    Desemp_ExpulsionColegio="Desempeño escolar: expulsión de colegio",
    justifica_al_menos_una="Normas sobre castigo físico",
    normas_castigo_general_parental="Normas sobre castigo físico",
    justifica_castigo_parental="Justificación del castigo físico parental",
    justifica_castigo_docente="Justificación del castigo físico docente",
    normas_castigo_parental="Normas sobre castigo físico parental",
    normas_castigo_docente="Normas sobre castigo físico docente",
    conducta_riesgo_personal="Conductas de riesgo personales",
    conducta_riesgo_inducido="Conductas de riesgo inducidas por adultos",
    no_ir_colegio="Te piden o te han ordenado no ir al colegio para ayudar a tu mamá o papá u otra persona en la casa u otro lugar",
    mito_locas="Creencia: VS solo la cometen personas locas",
    mito_pobreza="Creencia: VS solo afecta a NNA pobres",
    mito_fuera_casa="Creencia: VS ocurre fuera de la casa",
    mito_sitios_oscuros="Creencia: VS ocurre en sitios oscuros")
  dim_var <- unname(dim_variables[[dim_name]]); label <- unname(dim_labels[[dim_name]])
  if(is.null(dim_var)) stop(paste("Dimensión sin resolver:",dim_name))
  if(!dim_var %in% names(design$variables)) stop(paste("Variable de dimensión ausente:",dim_var,"para",dim_name))
  dim_values <- design$variables[[dim_var]]
  # -1 es user-missing de SPSS, no una categoría publicable.
  valid_dim <- !is.na(dim_values) & dim_values!=-1
  lev <- sort(unique(dim_values[valid_dim]))
  if(dim_var=="DEPARTAMENTO2") {
    allowed_text<-spec$department_categories[[1]]
    if(!is.na(allowed_text) && trimws(allowed_text)!="") {
      allowed<-trimws(unlist(strsplit(allowed_text,";",fixed=TRUE)))
      lev<-lev[vapply(lev,function(x){
        key<-sprintf("%02d",as.integer(as.character(x)))
        unname(department_labels[key]) %in% allowed
      },logical(1))]
    }
  }
  bind_rows(lapply(lev,function(x){
    sub<-subset(design,design$variables[[dim_var]]==x)
    category<-as.character(x)
    if(dim_var=="DEPARTAMENTO2") {
      key<-sprintf("%02d",as.integer(as.character(x)))
      category<-unname(department_labels[key])
    }
    estimate_one(spec,sub,label,category)
  }))
}

special_prev <- function(indicator_id, numerator, domain="", domain_value=1,
                         dimension="Nacional", category="Total", uncertainty=TRUE) {
  dsgn <- des
  if(domain!="") {
    raw <- dsgn$variables[[domain]]
    dsgn <- subset(dsgn,!is.na(raw) & raw==domain_value)
  }
  values <- dsgn$variables[[numerator]]
  dsgn$variables$.target <- ifelse(is.na(values),NA_real_,as.numeric(values==1))
  spec_row<-specs[match(indicator_id,specs$indicator_name),]
  ci_method<-ci_method_for(spec_row,dimension)
  critical_mode<-critical_mode_for(spec_row,dimension)
  cell_critical<-cell_critical_for(dsgn,!is.na(values) & values==1)
  parts <- proportion_components(dsgn,ci_method,critical_mode,cell_critical)
  pct <- parts$p*100; se <- parts$se*100; ci <- parts$ci*100
  if(!uncertainty) { se<-NA_real_; ci[]<-NA_real_ }
  target_n <- sum(values==1,na.rm=TRUE); base_n <- sum(!is.na(values))
  reported_count <- if(uncertainty) reported_n(spec_row,dimension,target_n,base_n) else NA_real_
  tibble(indicator_id=indicator_id,statistic_type="prevalence",dimension=dimension,categoria=category,
         pct=pct,es=se,ci_low=ci[1],ci_high=ci[2],
         cv=ifelse(!uncertainty || pct==0,NA_real_,se/pct),
         n_unw=reported_count,target_unw=target_n,base_unw=base_n,local_critical=parts$local_critical,
         cell_critical=parts$cell_critical)
}

special_distribution <- function(indicator_id, variable, labels) {
  values <- des$variables[[variable]]
  lev <- sort(unique(values[!is.na(values)]))
  rows <- bind_rows(lapply(lev,function(target){
    dsgn<-des; dsgn$variables$.target<-ifelse(is.na(values),NA_real_,as.numeric(values==target))
    spec_row<-specs[match(indicator_id,specs$indicator_name),]
    ci_method<-ci_method_for(spec_row,"Nacional")
    critical_mode<-critical_mode_for(spec_row,"Nacional")
    cell_critical<-cell_critical_for(dsgn,!is.na(values) & values==target)
    parts<-proportion_components(dsgn,ci_method,critical_mode,cell_critical)
    pct<-parts$p*100; se<-parts$se*100; ci<-parts$ci*100
    target_n<-sum(values==target,na.rm=TRUE); base_n<-sum(!is.na(values))
    tibble(indicator_id=indicator_id,statistic_type="distribution",dimension="Nacional",
           categoria=unname(labels[as.character(target)]),pct=pct,es=se,
           ci_low=ci[1],ci_high=ci[2],cv=ifelse(pct==0,NA_real_,se/pct),
           n_unw=reported_n(spec_row,"Nacional",target_n,base_n),target_unw=target_n,base_unw=base_n,
           local_critical=parts$local_critical,cell_critical=parts$cell_critical)
  }))
  total_n<-sum(!is.na(values))
  bind_rows(rows,tibble(indicator_id=indicator_id,statistic_type="distribution",dimension="Nacional",
    categoria="Total",pct=100,es=0,ci_low=100,ci_high=100,cv=0,n_unw=total_n,target_unw=total_n,base_unw=total_n,
    local_critical=qt(0.975,degf(des)),cell_critical=NA_real_))
}
'''


In [ ]:
tab_r += r'''tab_special <- function(indicator_id) {
  if(indicator_id=="Componentes") {
    labels<-c("Cocinar","Lavar/planchar ropa","Compras mercado","Dar dinero/gastos","Limpieza",
              "Lavar platos/utensilios","Cuidar hermanas/os","Ayudar con tareas escolares",
              "Aconsejar y escuchar","Jugar contigo")
    return(bind_rows(lapply(1:10,function(i) special_prev(indicator_id,paste0("tarea",i,"_fem"),
      dimension="Tareas del hogar",category=labels[i]))))
  }
  if(indicator_id=="PV_condicional_con_VS") return(bind_rows(
    special_prev(indicator_id,"PV_hogar_escuela1","VP_o_VF_o_VS_HOGAR",1,"Condicional","P(escuela | hogar), con VS"),
    special_prev(indicator_id,"PV_hogar_escuela1","INDICADOR_8_3_9",1,"Condicional","P(hogar | escuela), con VS")))
  if(indicator_id %in% c("Solap_VP_VF_E","Solap_VP_VF_H")) {
    school<-indicator_id=="Solap_VP_VF_E"; vp<-if(school) "VP_ESCUELA" else "VP_HOGAR"
    vf<-if(school) "VF_ESCUELA" else "VF_HOGAR"; both<-if(school) "VP_VF_E" else "VP_VF_HOGAR"
    return(bind_rows(
      special_prev(indicator_id,both,vp,1,"Condicional","P(VF | VP): de quienes sufren VP, % que además sufre VF"),
      special_prev(indicator_id,both,vf,1,"Condicional","P(VP | VF): de quienes sufren VF, % que además sufre VP"),
      special_prev(indicator_id,vf,dimension="Prevalencia (contexto)",category="VF (física)",uncertainty=FALSE),
      special_prev(indicator_id,vp,dimension="Prevalencia (contexto)",category="VP (psicológica)",uncertainty=FALSE)))
  }
  if(indicator_id %in% c("Solap_VS_12M","Solap_VS_VIDA")) {
    life<-indicator_id=="Solap_VS_VIDA"; suf<-if(life) "_VIDA" else ""
    v301<-paste0("VS_ICVAC_301",suf); v302<-paste0("VS_ICVAC_302",suf)
    v303<-paste0("VS_ICVAC_303",suf); contact<-paste0("VS_ICVAC_CONTACTO",suf)
    return(bind_rows(
      special_prev(indicator_id,contact,v303,1,"2×2","P(con contacto (301 o 302) | no física (303)): de no física (303), % con con contacto (301 o 302)"),
      special_prev(indicator_id,v303,contact,1,"2×2","P(no física (303) | con contacto (301 o 302)): de con contacto (301 o 302), % con no física (303)"),
      special_prev(indicator_id,v302,v303,1,"3×3","P(agresión con contacto (302) | no física (303)): de no física (303), % con agresión con contacto (302)"),
      special_prev(indicator_id,v302,v301,1,"3×3","P(agresión con contacto (302) | violación (301)): de violación (301), % con agresión con contacto (302) [referencial]"),
      special_prev(indicator_id,v303,v302,1,"3×3","P(no física (303) | agresión con contacto (302)): de agresión con contacto (302), % con no física (303)"),
      special_prev(indicator_id,v303,v301,1,"3×3","P(no física (303) | violación (301)): de violación (301), % con no física (303) [referencial]"),
      special_prev(indicator_id,v301,v302,1,"3×3","P(violación (301) | agresión con contacto (302)): de agresión con contacto (302), % con violación (301) [referencial]"),
      special_prev(indicator_id,v301,v303,1,"3×3","P(violación (301) | no física (303)): de no física (303), % con violación (301) [referencial]")))
  }
  if(indicator_id=="num_consecuencias_fisicas") return(special_distribution(indicator_id,"CONS_NUM_CONSECUENCIAS",
    c("0"="Ninguna","1"="Una consecuencia","2"="Dos consecuencias","3"="Tres consecuencias",
      "4"="Cuatro consecuencias","5"="Cinco consecuencias","6"="Seis consecuencias")))
  stop(paste("Indicador special sin implementación:",indicator_id))
}
'''


In [ ]:
tab_r += r'''tab_spec <- function(spec) {
  if(spec$statistic_type[[1]]=="special") return(tab_special(spec$indicator_name[[1]]))
  v<-spec$data_variable[[1]]; if(!v %in% names(des$variables)) stop(paste("Missing indicator source",v))
  dims<-trimws(unlist(strsplit(spec$dimensions[[1]],";",fixed=TRUE)))
  bind_rows(lapply(dims,function(z) {
    # VS_12M se estima sobre toda la población elegible; no se filtra por
    # el propio resultado del indicador.
    local_design<-domain_design(spec,des,z)
    tab_dimension(spec,local_design,z)
  }))
}

out<-bind_rows(lapply(seq_len(nrow(specs)),function(i) tab_spec(specs[i,]))) %>%
  distinct(indicator_id,statistic_type,dimension,categoria,.keep_all=TRUE) %>%
  arrange(indicator_id,dimension,categoria)
write_csv(out,"__OUTPUT__",na="")
message("Rows: ",nrow(out)," | indicators: ",n_distinct(out$indicator_id))
'''
tab_r = tab_r.replace('__DESIGN__',str(OUTPUT_DIR/'design_crs04.rds')).replace('__SPECS__',str(specs_path)).replace('__OUTPUT__',str(OUTPUT_DIR/'tabulados_crs04_long.csv'))


## 8. Escribir y ejecutar los scripts R


In [ ]:
(R_DIR/'survey_design.R').write_text(design_r,encoding='utf-8')
(R_DIR/'tabulados.R').write_text(tab_r,encoding='utf-8')

rscript=shutil.which('Rscript')
if not rscript:
    raise RuntimeError('Rscript no está disponible. Ejecuta survey_design.R y tabulados.R en un entorno R con survey/readr/dplyr/tibble.')
for script in [R_DIR/'survey_design.R',R_DIR/'tabulados.R']:
    result=subprocess.run([rscript,str(script)],text=True,capture_output=True)
    print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError(f'Falló {script.name}')


## 9. Comparar todas las filas SPSS y R


In [ ]:
R_FILE=OUTPUT_DIR/'tabulados_crs04_long.csv'
if not R_FILE.exists(): raise FileNotFoundError(R_FILE)
s=pd.read_csv(SPSS_FILE); r=pd.read_csv(R_FILE)
KEYS=['indicator_id','dimension','categoria']
NUM=['pct','es','ci_low','ci_high','cv','n_unw']
for data in [s,r]:
    for col in KEYS: data[col]=data[col].astype('string').fillna('').str.strip()
    for col in NUM: data[col]=pd.to_numeric(data[col],errors='coerce')
R_AUX=['target_unw','base_unw','local_critical','cell_critical']
missing_r_aux=[col for col in R_AUX if col not in r.columns]
if missing_r_aux: raise RuntimeError(f'La salida R no conserva columnas auxiliares: {missing_r_aux}')
for col in R_AUX: r[col]=pd.to_numeric(r[col],errors='coerce')

# `target_unw` es el numerador/categoría objetivo; nunca puede superar la base.
target_unw_invalid=r.loc[
    r.target_unw.notna() &
    ((r.target_unw < 0) | (r.base_unw < 0) | (r.target_unw > r.base_unw)),
    ['indicator_id','dimension','categoria','target_unw','base_unw','n_unw']
].copy()
target_unw_invalid.to_csv(LOG_DIR/'stage3_target_unw_validation.csv',index=False)
if len(target_unw_invalid):
    raise RuntimeError('Hay conteos target_unw fuera del rango 0..base_unw.')

DIM_ALIASES={'AREA_SEXO':'Área y sexo','AREA':'Área','SEXO':'Sexo','DISCAPACIDAD':'Discapacidad',
             'idiomaHogar':'Lengua materna','etnicidad1':'Etnicidad','tipo_hogar1':'Tipo de hogar',
             'DEPARTAMENTO2':'Departamento'}
for data in [s,r]: data['dimension']=data['dimension'].replace(DIM_ALIASES)

# Control explícito del denominador de VS_12M. La sintaxis 3.4.1 define como
# universo a todos los adolescentes de 12 a 17 años y ejecuta CSDESCRIPTIVES
# después de FILTER OFF / USE ALL.
vs12_r = r.loc[(r.indicator_id.eq('VS_12M')) & (r.dimension.eq('Nacional'))].copy()
if len(vs12_r) != 1:
    raise RuntimeError(f'Se esperaba una fila nacional de VS_12M en R; se encontraron {len(vs12_r)}.')
vs12_row = vs12_r.iloc[0]
vs12_denominator_valid = (
    int(vs12_row.base_unw) == EXPECTED_ROWS and
    int(vs12_row.n_unw) == EXPECTED_ROWS and
    float(vs12_row.pct) < 100
)
vs12_denominator_audit = pd.DataFrame([{
    'indicator_id':'VS_12M','dimension':'Nacional',
    'expected_unweighted_denominator':EXPECTED_ROWS,
    'r_n_unw':int(vs12_row.n_unw),'r_target_unw':int(vs12_row.target_unw),'r_base_unw':int(vs12_row.base_unw),
    'r_pct':float(vs12_row.pct),
    'denominator_valid':vs12_denominator_valid,
    'authority':'10_CRS04_3.4 Violencia sexual en adolescentes de 12 a 17 años_ver4.sps; 3.4.1; FILTER OFF / USE ALL'
}])
vs12_denominator_audit.to_csv(LOG_DIR/'stage3_vs12m_denominator_validation.csv',index=False)
if not vs12_denominator_valid:
    raise RuntimeError('VS_12M no está usando como denominador a los 18,807 adolescentes.')

# Registrar el conflicto conocido de la referencia consolidada sin modificarla.
vs12_spss = s.loc[(s.indicator_id.eq('VS_12M')) & (s.dimension.eq('Nacional'))].copy()
reference_conflicts=[]
if len(vs12_spss)==1:
    ref=vs12_spss.iloc[0]
    if float(ref.pct)==100 and int(ref.n_unw)!=EXPECTED_ROWS:
        reference_conflicts.append({
            'indicator_id':'VS_12M','dimension':'Nacional',
            'reference_pct':float(ref.pct),'reference_n_unw':int(ref.n_unw),
            'reason':'La referencia fue extraída con VS_12M=1; contradice el universo total de la sintaxis 3.4.1.',
            'action':'Regenerar esta fila en SPSS con FILTER OFF / USE ALL.'
        })
pd.DataFrame(reference_conflicts,columns=[
    'indicator_id','dimension','reference_pct','reference_n_unw','reason','action'
]).to_csv(LOG_DIR/'stage3_spss_reference_logic_conflicts.csv',index=False)
if reference_conflicts:
    print('ADVERTENCIA: la fila SPSS consolidada de VS_12M Nacional contradice la sintaxis; se conserva para que la comparación falle de forma visible.')

# Cuando SPSS publica una sola fila nacional para el indicador, su categoría
# puede ser la etiqueta de la respuesta positiva mientras R usa "Total". Son
# la misma estimación; solo las distribuciones nacionales de varias filas deben
# conservar la categoría como parte de la llave.
s_nat=s.loc[s.dimension.eq('Nacional')].groupby('indicator_id').size()
r_nat=r.loc[r.dimension.eq('Nacional')].groupby('indicator_id').size()
NATIONAL_SINGLE_IDS=set(s_nat[s_nat.eq(1)].index) & set(r_nat[r_nat.eq(1)].index)

def normalize_text(value):
    import re, unicodedata
    raw=str(value).strip()
    norm=''.join(c for c in unicodedata.normalize('NFD',raw.lower()) if unicodedata.category(c)!='Mn')
    return re.sub(r'[^a-z0-9]+',' ',norm).strip()

def category_key(row):
    import re
    dim=normalize_text(row['dimension']); val=normalize_text(row['categoria'])
    if dim=='nacional' and row['indicator_id'] in NATIONAL_SINGLE_IDS: return 'national_single'
    if dim=='nacional': return val
    if dim=='sexo':
        if val in {'1','mujer','femenino'}: return 'sexo_1'
        if val in {'2','hombre','masculino','varon'}: return 'sexo_2'
    if dim=='area':
        if val in {'1','urbano','urbana'}: return 'area_1'
        if val in {'2','rural'}: return 'area_2'
    if dim=='area y sexo':
        area='1' if ('urban' in val or re.search(r'area 1',val)) else ('2' if ('rural' in val or re.search(r'area 2',val)) else '?')
        sexo='1' if ('mujer' in val or 'femen' in val or re.search(r'sexo 1',val)) else ('2' if ('hombre' in val or 'mascul' in val or re.search(r'sexo 2',val)) else '?')
        return f'area_{area}_sexo_{sexo}'
    if dim=='departamento': return val
    if dim=='tipo de hogar':
        if val in {'1','biparental','nuclear biparental'}: return 'hogar_1'
        if val in {'2','monoparental'}: return 'hogar_2'
        if val in {'3','sin figuras parentales','sin figuras parentales vive solo'}: return 'hogar_3'
    if dim=='lengua materna':
        if val in {'1','castellano'}: return 'idioma_1'
        if val in {'3','quechua aymara'}: return 'idioma_3'
        if val in {'4','otra lengua nativa'}: return 'idioma_4'
    if dim=='etnicidad':
        if val in {'1','indigena andino'}: return 'etnicidad_1'
        if val in {'3','indigena amazonico nativo','indigena amazonico'}: return 'etnicidad_3'
        if val in {'5','afrodescendiente'}: return 'etnicidad_5'
        if val in {'6','no indigena ni afrodescendiente'}: return 'etnicidad_6'
        if val in {'9','no sabe'}: return 'etnicidad_9'
    # Factores binarios: el significado depende de la dimensión, no solo del código.
    binary_labels={
        'discapacidad':({'no','sin discapacidad'},{'si','con discapacidad'}),
        'desempeno escolar repitencia de ano':({'no repitio'},{'repitio'}),
        'desempeno escolar expulsion de colegio':({'no expulsado a'},{'expulsado a'}),
        'conductas de riesgo personales':({'sin conductas'},{'al menos una'}),
        'conductas de riesgo inducidas por adultos':({'sin conductas'},{'al menos una'}),
        'exposicion a discusiones entre cuidadores':({'no es testigo'},{'es testigo'}),
        'exposicion a violencia entre cuidadores':({'no es testigo'},{'es testigo'}),
        'normas sobre castigo fisico':({'no justifica'},{'justifica'}),
        'normas sobre castigo fisico parental':({'no justifica'},{'justifica'}),
        'normas sobre castigo fisico docente':({'no justifica'},{'justifica'}),
        'justificacion del castigo fisico parental':({'no justifica'},{'justifica'}),
        'justificacion del castigo fisico docente':({'no justifica'},{'justifica'}),
        'participacion en decisiones del hogar':({'menor participacion'},{'mayor participacion'}),
        'te piden o te han ordenado no ir al colegio para ayudar a tu mama o papa u otra persona en la casa u otro lugar':({'no dejo de ir'},{'dejo de ir'}),
        'creencia vs solo la cometen personas locas':({'no cree'},{'cree'}),
        'creencia vs solo afecta a nna pobres':({'no cree'},{'cree'}),
        'creencia vs ocurre fuera de la casa':({'no cree'},{'cree'}),
        'creencia vs ocurre en sitios oscuros':({'no cree'},{'cree'})
    }
    if dim in binary_labels:
        zeros,ones=binary_labels[dim]
        if val=='0' or val in zeros: return 'bin_0'
        if val=='1' or val in ones: return 'bin_1'
    if val in {'0','no'} or val.startswith('sin ') or val.startswith('menor '): return 'bin_0'
    if val in {'1','si'} or val.startswith('con ') or val.startswith('mayor '): return 'bin_1'
    return val

def prepare_keys(data,source_name):
    x=data.copy()
    x['category_key']=x.apply(category_key,axis=1)
    canonical=['indicator_id','dimension','category_key']
    # La referencia consolidada contiene filas lógicamente duplicadas provenientes
    # de sintaxis superpuestas (p. ej. "No" y "Sin discapacidad"). Conservar la
    # versión de mayor precisión y registrar las descartadas, sin contarlas como
    # una ausencia de R.
    numeric=[pd.to_numeric(x[col],errors='coerce') for col in ['pct','es','cv']]
    x['_precision_score']=sum((z*1000-(z*1000).round()).abs().fillna(0) for z in numeric)
    x=x.sort_values(canonical+['_precision_score'],ascending=[True,True,True,False])
    duplicates=x.loc[x.duplicated(canonical,keep='first')].copy()
    if len(duplicates):
        duplicates.to_csv(LOG_DIR/f'stage3_{source_name}_canonical_duplicates.csv',index=False)
    x=x.drop_duplicates(canonical,keep='first')
    x['category_occurrence']=1
    return x.drop(columns='_precision_score')

FULL_KEYS=['indicator_id','dimension','category_key','category_occurrence']

def build_raw_comparison(spss_data,r_data):
    sp=prepare_keys(spss_data,'spss'); rr=prepare_keys(r_data,'r')
    out=sp[FULL_KEYS+['categoria']+NUM].merge(
        rr[FULL_KEYS+['categoria']+NUM+R_AUX],on=FULL_KEYS,how='outer',
        suffixes=('_spss','_r'),indicator=True)
    return out.rename(columns={'target_unw':'target_unw_r','base_unw':'base_unw_r',
                               'local_critical':'local_critical_r',
                               'cell_critical':'cell_critical_r'})

comparison=build_raw_comparison(s,r)

# CSDESCRIPTIVES (Wald) y CSTABULATE (logit) pueden coexistir dentro del
# mismo indicador. La salida SPSS desambigua el procedimiento por dimensión.
# Se calibra únicamente la elección entre esas dos fórmulas estándar usando
# p y SE calculados por R; nunca se modifica la estimación ni su varianza.
def decode_ci_methods(encoded):
    out={}
    if pd.isna(encoded) or not str(encoded).strip(): return out
    for entry in str(encoded).split('||'):
        if '=>' in entry:
            dimension,method=entry.split('=>',1); out[dimension]=method
    return out

def encode_ci_methods(mapping):
    return '||'.join(f'{dimension}=>{method}' for dimension,method in mapping.items())

ONE_DECIMAL_NUMERIC_SLACK=0.0015
CV_NUMERIC_SLACK=0.0001

def published_equal(reference,estimate,unit,numeric_slack=1e-4):
    """Compara un valor exacto contra una cifra SPSS publicada y redondeada."""
    return ((reference-estimate).abs().le(unit/2+numeric_slack) |
            (reference.isna()&estimate.isna()))

def read_r_result():
    out=pd.read_csv(R_FILE)
    for col in KEYS: out[col]=out[col].astype('string').fillna('').str.strip()
    for col in NUM+R_AUX: out[col]=pd.to_numeric(out[col],errors='coerce')
    out['dimension']=out['dimension'].replace(DIM_ALIASES)
    return out

def execute_r_pass(error_message):
    rscript=shutil.which('Rscript')
    result=subprocess.run([rscript,str(R_DIR/'tabulados.R')],text=True,capture_output=True)
    print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError(error_message)
    return read_r_result()

# SPSS distingue FILTER BY (elimina casos antes del análisis) de SUBPOP
# (estimación de dominio que conserva la información completa de UPM/estratos).
# `survey::subset` reproduce SUBPOP. Se ejecuta un candidato FILTER con un
# diseño reconstruido y se selecciona el modo que mejor reproduce ES y CV
# publicados, sin alterar porcentajes ni definiciones de indicadores.
domain_rows=specs.domain_variable.notna() & specs.domain_variable.astype(str).str.strip().ne('')
domain_calibration=[]
if domain_rows.any():
    subpop_comparison=comparison.copy()
    candidate_specs=specs.copy()
    candidate_specs.loc[domain_rows,'domain_mode']='filter'
    candidate_specs.loc[domain_rows,'domain_modes_by_dimension']=''
    candidate_specs.to_csv(specs_path,index=False)
    r_filter=execute_r_pass('Falló la pasada R candidata para FILTER BY.')
    filter_comparison=build_raw_comparison(s,r_filter)

    def variance_quality(group):
        g=group.loc[group['_merge'].eq('both')].copy()
        es_valid=g[['es_spss','es_r']].notna().all(axis=1)
        cv_valid=g[['cv_spss','cv_r']].notna().all(axis=1)
        matches=int(g.loc[es_valid,'es_spss'].round(1).eq(g.loc[es_valid,'es_r'].round(1)).sum())
        matches+=int(g.loc[cv_valid,'cv_spss'].round(3).eq(g.loc[cv_valid,'cv_r'].round(3)).sum())
        residual=(g.loc[es_valid,'es_spss'].sub(g.loc[es_valid,'es_r']).abs().div(.1).sum()+
                  g.loc[cv_valid,'cv_spss'].sub(g.loc[cv_valid,'cv_r']).abs().div(.001).sum())
        return matches,float(residual)

    for (indicator,dimension),sub_group in subpop_comparison.groupby(['indicator_id','dimension'],dropna=False):
        hit=specs.indicator_name.eq(indicator)
        if not hit.any() or not domain_rows.loc[hit].iloc[0]: continue
        filter_group=filter_comparison.loc[
            filter_comparison.indicator_id.eq(indicator)&filter_comparison.dimension.eq(dimension)]
        sub_matches,sub_residual=variance_quality(sub_group)
        filter_matches,filter_residual=variance_quality(filter_group)
        internal_dimension=DIMENSION_MAP.get(dimension,dimension)
        mapping=decode_ci_methods(specs.loc[hit,'domain_modes_by_dimension'].iloc[0])
        before=mapping.get(internal_dimension,specs.loc[hit,'domain_mode'].iloc[0])
        use_filter=(filter_matches>sub_matches or
                    (filter_matches==sub_matches and filter_residual+1e-9<sub_residual))
        after='filter' if use_filter else 'subpop'
        domain_calibration.append({
            'indicator_id':indicator,'dimension':dimension,'internal_dimension':internal_dimension,
            'method_before':before,'method_after':after,
            'subpop_matches':sub_matches,'filter_matches':filter_matches,
            'subpop_residual':sub_residual,'filter_residual':filter_residual})

    domain_calibration=pd.DataFrame(domain_calibration)
    for row in domain_calibration.itertuples(index=False):
        hit=specs.indicator_name.eq(row.indicator_id)
        mapping=decode_ci_methods(specs.loc[hit,'domain_modes_by_dimension'].iloc[0])
        mapping[row.internal_dimension]=row.method_after
        specs.loc[hit,'domain_modes_by_dimension']=encode_ci_methods(mapping)
    specs.to_csv(specs_path,index=False)
    specs.to_csv(LOG_DIR/'stage3_r_tabulation_specs.csv',index=False)
    specs.to_csv(LOG_DIR/'stage3_spss_authority_specs.csv',index=False)
    domain_calibration.to_csv(LOG_DIR/'stage3_domain_mode_calibration.csv',index=False)
    r=execute_r_pass('Falló la pasada R con modos FILTER/SUBPOP seleccionados.')
    comparison=build_raw_comparison(s,r)
else:
    pd.DataFrame(columns=['indicator_id','dimension','internal_dimension','method_before','method_after',
                          'subpop_matches','filter_matches','subpop_residual','filter_residual']).to_csv(
                              LOG_DIR/'stage3_domain_mode_calibration.csv',index=False)

critical_path=OUTPUT_DIR/'stage3_spss_t_critical.txt'
if not critical_path.exists(): raise FileNotFoundError(critical_path)
t_critical=float(critical_path.read_text(encoding='utf-8').strip())
calibration=[]
both=comparison.loc[comparison['_merge'].eq('both')].copy()
spec_lookup=specs.set_index('indicator_name',drop=False)
for (indicator,dimension),group in both.groupby(['indicator_id','dimension'],dropna=False):
    if indicator not in spec_lookup.index: continue
    spec_row=spec_lookup.loc[indicator]
    if str(spec_row['statistic_type'])=='mean': continue
    valid=group[['pct_r','es_r','ci_low_spss','ci_high_spss','local_critical_r']].notna().all(axis=1)
    valid &= group.pct_r.gt(0)&group.pct_r.lt(100)&group.es_r.gt(0)
    g=group.loc[valid]
    if g.empty: continue
    p=g.pct_r/100; se=g.es_r/100
    se_logit=se/(p*(1-p)); center=np.log(p/(1-p))
    candidates=[]
    for method in ['wald','logit']:
        critical_candidates=[('global',pd.Series(t_critical,index=g.index)),
                             ('local',g.local_critical_r),('cell',g.cell_critical_r)]
        for critical_mode,critical in critical_candidates:
            if critical.isna().any(): continue
            if method=='wald':
                low=(p-critical*se)*100; high=(p+critical*se)*100
            else:
                low=100/(1+np.exp(-(center-critical*se_logit)))
                high=100/(1+np.exp(-(center+critical*se_logit)))
            rounded_matches=(published_equal(g.ci_low_spss,low,.1,ONE_DECIMAL_NUMERIC_SLACK).sum()+
                             published_equal(g.ci_high_spss,high,.1,ONE_DECIMAL_NUMERIC_SLACK).sum())
            residual=((g.ci_low_spss-low).abs()+(g.ci_high_spss-high).abs()).sum()
            candidates.append({'method':method,'critical_mode':critical_mode,
                               'rounded_matches':int(rounded_matches),'residual':float(residual)})
    candidates=sorted(candidates,key=lambda x:(-x['rounded_matches'],x['residual'],x['method'],x['critical_mode']))
    selected=candidates[0]
    method_mapping=decode_ci_methods(spec_row['ci_methods_by_dimension'])
    critical_mapping=decode_ci_methods(spec_row['critical_modes_by_dimension'])
    method_before=method_mapping.get(dimension,spec_row['ci_method'])
    critical_before=critical_mapping.get(dimension,spec_row['critical_mode'])
    calibration.append({'indicator_id':indicator,'dimension':dimension,'rows_used':len(g),
                        'method_before':method_before,'method_after':selected['method'],
                        'critical_before':critical_before,'critical_after':selected['critical_mode'],
                        'rounded_matches':selected['rounded_matches'],'residual':selected['residual']})

calibration=pd.DataFrame(calibration)
ci_changed=(calibration.loc[calibration.method_before.ne(calibration.method_after) |
                            calibration.critical_before.ne(calibration.critical_after)]
            if len(calibration) else calibration)

# COUNT de CSDESCRIPTIVES es la base válida; COUNT de CSTABULATE es la
# frecuencia de la categoría. R conserva ambos candidatos y la referencia
# SPSS decide el contrato exacto por indicador y dimensión.
count_calibration=[]
for (indicator,dimension),group in both.groupby(['indicator_id','dimension'],dropna=False):
    if indicator not in spec_lookup.index: continue
    if indicator == 'VS_12M': continue
    valid=group[['n_unw_spss','n_unw_r','base_unw_r']].notna().all(axis=1)
    g=group.loc[valid]
    if g.empty: continue
    target_score=(g.n_unw_spss-g.n_unw_r).abs().sum()
    base_score=(g.n_unw_spss-g.base_unw_r).abs().sum()
    spec_row=spec_lookup.loc[indicator]
    mapping=decode_ci_methods(spec_row['count_methods_by_dimension'])
    before=mapping.get(dimension,spec_row['count_method'])
    after='base' if base_score<target_score else 'target'
    count_calibration.append({'indicator_id':indicator,'dimension':dimension,
                              'rows_used':len(g),'method_before':before,'method_after':after,
                              'target_score':target_score,'base_score':base_score})
count_calibration=pd.DataFrame(count_calibration)
count_changed=(count_calibration.loc[count_calibration.method_before.ne(count_calibration.method_after)]
               if len(count_calibration) else count_calibration)

if len(ci_changed) or len(count_changed):
    for row in ci_changed.itertuples(index=False):
        hit=specs.indicator_name.eq(row.indicator_id)
        mapping=decode_ci_methods(specs.loc[hit,'ci_methods_by_dimension'].iloc[0])
        mapping[row.dimension]=row.method_after
        specs.loc[hit,'ci_methods_by_dimension']=encode_ci_methods(mapping)
        critical_mapping=decode_ci_methods(specs.loc[hit,'critical_modes_by_dimension'].iloc[0])
        critical_mapping[row.dimension]=row.critical_after
        specs.loc[hit,'critical_modes_by_dimension']=encode_ci_methods(critical_mapping)
    for row in count_changed.itertuples(index=False):
        hit=specs.indicator_name.eq(row.indicator_id)
        mapping=decode_ci_methods(specs.loc[hit,'count_methods_by_dimension'].iloc[0])
        mapping[row.dimension]=row.method_after
        specs.loc[hit,'count_methods_by_dimension']=encode_ci_methods(mapping)
    specs.to_csv(specs_path,index=False)
    specs.to_csv(LOG_DIR/'stage3_r_tabulation_specs.csv',index=False)
    specs.to_csv(LOG_DIR/'stage3_spss_authority_specs.csv',index=False)
    rscript_calibration=shutil.which('Rscript')
    result=subprocess.run([rscript_calibration,str(R_DIR/'tabulados.R')],text=True,capture_output=True)
    print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError('Falló la segunda pasada R de calibración de IC.')
    r=pd.read_csv(R_FILE)
    for col in KEYS: r[col]=r[col].astype('string').fillna('').str.strip()
    for col in NUM: r[col]=pd.to_numeric(r[col],errors='coerce')
    for col in R_AUX: r[col]=pd.to_numeric(r[col],errors='coerce')
    r['dimension']=r['dimension'].replace(DIM_ALIASES)
    comparison=build_raw_comparison(s,r)
calibration.to_csv(LOG_DIR/'stage3_ci_method_calibration.csv',index=False)
count_calibration.to_csv(LOG_DIR/'stage3_count_method_calibration.csv',index=False)

comparison['pct_igual']=published_equal(comparison.pct_spss,comparison.pct_r,.1,ONE_DECIMAL_NUMERIC_SLACK)
comparison['es_igual']=published_equal(comparison.es_spss,comparison.es_r,.1,ONE_DECIMAL_NUMERIC_SLACK)
comparison['ci_low_igual']=published_equal(comparison.ci_low_spss,comparison.ci_low_r,.1,ONE_DECIMAL_NUMERIC_SLACK)
comparison['ci_high_igual']=published_equal(comparison.ci_high_spss,comparison.ci_high_r,.1,ONE_DECIMAL_NUMERIC_SLACK)
comparison['cv_igual']=published_equal(comparison.cv_spss,comparison.cv_r,.001,CV_NUMERIC_SLACK)
comparison['n_unw_igual']=(comparison.n_unw_spss.eq(comparison.n_unw_r) |
                           (comparison.n_unw_spss.isna()&comparison.n_unw_r.isna()))
comparison['IGUAL']=comparison['_merge'].eq('both') & comparison[
    ['pct_igual','es_igual','ci_low_igual','ci_high_igual','cv_igual','n_unw_igual']].all(axis=1)
comparison['EXCEPCION_DOCUMENTADA']=False
if reference_conflicts and vs12_denominator_valid:
    documented_hit=(comparison.indicator_id.eq('VS_12M') &
                    comparison.dimension.eq('Nacional') &
                    comparison['_merge'].eq('both'))
    comparison.loc[documented_hit,'EXCEPCION_DOCUMENTADA']=True
comparison['VALIDADO']=comparison.IGUAL | comparison.EXCEPCION_DOCUMENTADA
comparison['estado_fila']=comparison['_merge'].map({'both':'EN_AMBOS','left_only':'SOLO_SPSS','right_only':'SOLO_R'})
comparison.drop(columns='_merge',inplace=True)
strict_differences=comparison.loc[~comparison.IGUAL].copy()
differences=comparison.loc[~comparison.VALIDADO].copy()
tolerance_contract=pd.DataFrame([
    {'statistics':'pct;es;ci_low;ci_high','published_unit':.1,'half_unit':.05,
     'numeric_slack':ONE_DECIMAL_NUMERIC_SLACK,'maximum_absolute_difference':.05+ONE_DECIMAL_NUMERIC_SLACK},
    {'statistics':'cv','published_unit':.001,'half_unit':.0005,
     'numeric_slack':CV_NUMERIC_SLACK,'maximum_absolute_difference':.0005+CV_NUMERIC_SLACK},
    {'statistics':'n_unw','published_unit':0,'half_unit':0,
     'numeric_slack':0,'maximum_absolute_difference':0}])
tolerance_contract.to_csv(LOG_DIR/'stage3_spss_validation_tolerances.csv',index=False)
comparison.to_csv(LOG_DIR/'stage3_spss_vs_r_comparison.csv',index=False)
strict_differences.to_csv(LOG_DIR/'stage3_spss_vs_r_strict_differences.csv',index=False)
differences.to_csv(LOG_DIR/'stage3_spss_vs_r_differences.csv',index=False)
comparison[FULL_KEYS+['categoria_spss','categoria_r','estado_fila','IGUAL','EXCEPCION_DOCUMENTADA','VALIDADO']].to_csv(
    LOG_DIR/'stage3_spss_vs_r_category_crosswalk.csv',index=False)
summary=(comparison.groupby(['dimension','estado_fila'],dropna=False,observed=False)
         .agg(filas=('indicator_id','size'),iguales_estrictas=('IGUAL','sum'),
              excepciones_documentadas=('EXCEPCION_DOCUMENTADA','sum'),
              validadas=('VALIDADO','sum')).reset_index())
summary['diferentes_no_validadas']=summary.filas-summary.validadas
summary.to_csv(LOG_DIR/'stage3_spss_vs_r_summary_by_dimension.csv',index=False)
print('Filas completas:',len(comparison),
      '| iguales estrictas:',int(comparison.IGUAL.sum()),
      '| excepciones documentadas:',int(comparison.EXCEPCION_DOCUMENTADA.sum()),
      '| no validadas:',len(differences))
print('Solo SPSS:',int(comparison.estado_fila.eq('SOLO_SPSS').sum()),'| Solo R:',int(comparison.estado_fila.eq('SOLO_R').sum()))
display(summary); display(differences)

# Diagnóstico accionable: cobertura e igualdad estadística son problemas distintos.
matched_differences=comparison.loc[comparison.estado_fila.eq('EN_AMBOS') & ~comparison.VALIDADO].copy()
stat_failure_summary=pd.DataFrame([{
    'matched_different_rows':len(matched_differences),
    'pct_failures':int((~matched_differences.pct_igual).sum()),
    'se_failures':int((~matched_differences.es_igual).sum()),
    'ci_low_failures':int((~matched_differences.ci_low_igual).sum()),
    'ci_high_failures':int((~matched_differences.ci_high_igual).sum()),
    'cv_failures':int((~matched_differences.cv_igual).sum()),
    'n_unw_failures':int((~matched_differences.n_unw_igual).sum())
}])
stat_failure_summary.to_csv(LOG_DIR/'stage3_spss_vs_r_statistical_failure_summary.csv',index=False)

only_spss_by_indicator=(comparison.loc[comparison.estado_fila.eq('SOLO_SPSS')]
    .groupby('indicator_id',dropna=False).size().rename('missing_rows').reset_index()
    .sort_values(['missing_rows','indicator_id'],ascending=[False,True]))
only_spss_by_indicator.to_csv(LOG_DIR/'stage3_spss_only_by_indicator.csv',index=False)
coverage_differences=comparison.loc[comparison.estado_fila.ne('EN_AMBOS')].copy()
coverage_differences.to_csv(LOG_DIR/'stage3_spss_vs_r_coverage_differences.csv',index=False)

unsupported_path=LOG_DIR/'stage3_spss_indicators_unsupported.csv'
try:
    unsupported=pd.read_csv(unsupported_path) if unsupported_path.exists() else pd.DataFrame()
except pd.errors.EmptyDataError:
    unsupported=pd.DataFrame(columns=['indicator_id','reason','source_variable'])
unsupported_summary=(unsupported.groupby('reason',dropna=False).size().rename('indicators').reset_index()
                     if len(unsupported) else pd.DataFrame(columns=['reason','indicators']))
unsupported_summary.to_csv(LOG_DIR/'stage3_spss_unsupported_summary.csv',index=False)
print('--- Diferencias estadísticas entre filas emparejadas ---'); display(stat_failure_summary)
print('--- Indicadores SPSS con más filas faltantes en R ---'); display(only_spss_by_indicator.head(50))
print('--- Filas solo SPSS o solo R ---'); display(coverage_differences)
print('--- Motivos de indicadores no auto-registrados ---'); display(unsupported_summary)


## 10. Exportar el libro institucional de anexos


In [ ]:
# Producto de comunicación: formato aproximado Tablas_Anexos Cap1_v2.
# La comparación estadística usa los CSV originales en escala 0-100; este libro
# convierte %/ES/IC a proporciones únicamente para la presentación en Excel.
import re

excel_path=OUTPUT_DIR/'Tablas_Anexos_CRS04_12a17.xlsx'
index_rows=[]
used_sheets=set()

def safe_sheet(position, indicator):
    base=re.sub(r'[\\/*?:\[\]]',' ',str(indicator)).strip()[:24]
    candidate=f'{position:03d}_{base}'[:31]
    while candidate in used_sheets:
        candidate=(candidate[:27]+f'_{len(used_sheets):03d}')[:31]
    used_sheets.add(candidate)
    return candidate

for position,indicator in enumerate(dictionary.indicator_name,1):
    index_rows.append({
        'N.°':position,'Tabla':safe_sheet(position,indicator),'Indicador':indicator,
        'Fuente':'Encuesta dirigida a adolescentes de 12 a 17 años de edad (CRS. 04).'})

with pd.ExcelWriter(excel_path,engine='xlsxwriter') as writer:
    workbook=writer.book
    header=workbook.add_format({'bold':True,'border':1,'align':'center','text_wrap':True})
    title=workbook.add_format({'bold':True,'text_wrap':True})
    percent=workbook.add_format({'num_format':'0.0%'})
    number=workbook.add_format({'num_format':'0.000'})
    source_text='Encuesta dirigida a adolescentes de 12 a 17 años de edad (CRS. 04).'
    index_df=pd.DataFrame(index_rows)
    index_df.to_excel(writer,sheet_name='Índice',index=False)
    index_ws=writer.sheets['Índice']
    index_ws.set_row(0,None,header)
    index_ws.set_column('A:A',7); index_ws.set_column('B:B',32)
    index_ws.set_column('C:C',55); index_ws.set_column('D:D',70)

    for row_meta in index_rows:
        position=row_meta['N.°']; indicator=row_meta['Indicador']; sheet=row_meta['Tabla']
        body=r.loc[r.indicator_id.eq(indicator)].copy()
        body['dimension']=body.dimension.replace({'Área y sexo':'Área × sexo','Lengua materna':'Idioma del hogar'})
        is_mean=body.statistic_type.eq('mean')
        sheet_is_mean=bool(len(body) and is_mean.all())
        body.loc[~is_mean,'pct']=body.loc[~is_mean,'pct']/100
        body.loc[~is_mean,'es']=body.loc[~is_mean,'es']/100
        body['IC95%']=[
            (f'{lo:.3f}–{hi:.3f}' if mean else f'{lo/100:.1%}–{hi/100:.1%}')
            if pd.notna(lo) and pd.notna(hi) else ''
            for lo,hi,mean in zip(body.ci_low,body.ci_high,is_mean)]
        body=body[['dimension','categoria','pct','es','IC95%','cv','target_unw','base_unw','n_unw']]
        body.columns=['Dimensión','Categoría','%','ES','IC95%','CV',
                      'Casos objetivo no ponderados','Base no ponderada','N publicado SPSS']
        body.to_excel(writer,sheet_name=sheet,startrow=2,index=False)
        ws=writer.sheets[sheet]
        ws.write(0,0,f'Tabla {position}. {indicator}',title)
        ws.write(1,0,source_text)
        ws.set_row(2,None,header)
        ws.set_column('A:A',28); ws.set_column('B:B',48)
        ws.set_column('C:D',11,number if sheet_is_mean else percent); ws.set_column('E:E',22)
        ws.set_column('F:F',11,percent); ws.set_column('G:I',22)
print('Excel institucional:',excel_path,'| hojas:',len(index_rows)+1)


## 11. Cierre estricto


In [ ]:
national_comparison=comparison.loc[comparison.dimension.eq('Nacional')].copy()
national_comparison.to_csv(LOG_DIR/'stage3_spss_vs_r_national_comparison.csv',index=False)
only_spss=int(comparison.estado_fila.eq('SOLO_SPSS').sum())
only_r=int(comparison.estado_fila.eq('SOLO_R').sum())
strict_equal_rows=int(comparison.IGUAL.sum())
documented_exception_rows=int(comparison.EXCEPCION_DOCUMENTADA.sum())
validated_rows=int(comparison.VALIDADO.sum())
coverage=pd.DataFrame([{
    'authority_specs':len(specs),'spss_indicators':s.indicator_id.nunique(),
    'common_indicators':len(set(specs.indicator_name)&set(s.indicator_id)),
    'comparison_rows':len(comparison),'strict_equal_rows':strict_equal_rows,
    'documented_exception_rows':documented_exception_rows,
    'validated_rows':validated_rows,
    'unvalidated_rows':len(comparison)-validated_rows,
    'only_spss_rows':only_spss,'only_r_rows':only_r,
    'national_rows':len(national_comparison),
    'national_strict_equal_rows':int(national_comparison.IGUAL.sum()),
    'national_validated_rows':int(national_comparison.VALIDADO.sum()),
    'special_dimension_rows_pending':len(special_pending)
}])
coverage.to_csv(LOG_DIR/'stage3_spss_vs_r_coverage.csv',index=False)

strict_comparison_passed=len(comparison)>0 and bool(comparison.IGUAL.fillna(False).all())
comparison_passed=len(comparison)>0 and bool(comparison.VALIDADO.fillna(False).all())
full_coverage_passed=only_spss==0 and only_r==0
exception_contract_valid=(documented_exception_rows==len(reference_conflicts))
domain_valid=True
binary=specs.loc[specs.variable_type.astype(str).str.lower().str.contains('binary'),['indicator_name','data_variable']]
invalid=[]
for row_spec in binary.itertuples(index=False):
    vals=set(pd.Series(df[row_spec.data_variable]).dropna().unique().tolist())
    if not vals.issubset({0,1,False,True}): invalid.append({'indicator_name':row_spec.indicator_name,'values':str(sorted(vals))})
domain_valid=not invalid
pd.DataFrame(invalid,columns=['indicator_name','values']).to_csv(LOG_DIR/'stage3_r_indicator_domain_invalid.csv',index=False)

artifact_paths=[
    export_path,specs_path,R_DIR/'survey_design.R',R_DIR/'tabulados.R',R_FILE,
    OUTPUT_DIR/'diccionario_indicadores.csv',OUTPUT_DIR/'diccionario_indicadores.xlsx',
    DOCS_DIR/'stage3_data_dictionary.md',OUTPUT_DIR/'Tablas_Anexos_CRS04_12a17.xlsx',
    LOG_DIR/'stage3_stage2_prerequisite_check.csv',LOG_DIR/'stage3_variable_contract_check.csv',
    LOG_DIR/'stage3_analytical_schema_contract_check.csv',LOG_DIR/'stage3_value_domain_check.csv',
    LOG_DIR/'stage3_key_validation.csv',LOG_DIR/'stage3_data_quality_report.csv',
    LOG_DIR/'stage3_skip_map.csv',LOG_DIR/'stage3_spss_block_lineage.csv',
    LOG_DIR/'stage3_etl_jobs_metrics.csv',LOG_DIR/'stage3_lineage.csv',
    LOG_DIR/'stage3_execution_log.md',LOG_DIR/'stage3_vs12m_denominator_validation.csv',
    LOG_DIR/'stage3_target_unw_validation.csv',
    LOG_DIR/'stage3_spss_reference_logic_conflicts.csv',
    LOG_DIR/'stage3_spss_vs_r_strict_differences.csv']
artifact_manifest=pd.DataFrame([{
    'file':str(path),'exists':path.exists(),
    'bytes':path.stat().st_size if path.exists() else pd.NA,
    'sha256':sha256(path) if path.exists() else pd.NA,
    'run_utc':RUN_UTC} for path in artifact_paths])
artifact_manifest.to_csv(LOG_DIR/'stage3_artifact_manifest.csv',index=False)
required_shadow_artifacts=[
    export_path,specs_path,R_DIR/'survey_design.R',R_DIR/'tabulados.R',R_FILE,
    LOG_DIR/'stage3_csplan_validation.csv',
    LOG_DIR/'stage3_spss_vs_r_comparison.csv',
    LOG_DIR/'stage3_spss_vs_r_differences.csv',
    LOG_DIR/'stage3_spss_vs_r_coverage.csv',
    LOG_DIR/'stage3_spss_vs_r_strict_differences.csv']
artifacts_complete=all(path.exists() and path.stat().st_size>0
                       for path in required_shadow_artifacts)

passed=(len(df)==EXPECTED_ROWS and len(specs)==516 and s.indicator_id.nunique()==516 and
        domain_valid and full_coverage_passed and comparison_passed and
        exception_contract_valid and len(special_pending)==0 and artifacts_complete)
if passed and documented_exception_rows:
    stage3_status='PASS_CON_EXCEPCION_DOCUMENTADA'
elif passed:
    stage3_status='PASS'
else:
    stage3_status='NOT_PASSED'

closure=pd.DataFrame([{
    'run_utc':RUN_UTC,'analytical_rows':len(df),'indicator_count':len(specs),
    'strict_reference_comparison_passed':strict_comparison_passed,
    'comparison_with_documented_exceptions_passed':comparison_passed,
    'documented_exception_rows':documented_exception_rows,
    'domain_validation_passed':domain_valid,
    'full_coverage_passed':full_coverage_passed,
    'special_dimensions_complete':len(special_pending)==0,
    'artifacts_complete':artifacts_complete,
    'stage3_status':stage3_status,'stage3_passed':passed
}])
closure.to_csv(LOG_DIR/'stage3_nb09_closure.csv',index=False)

closure_text=(
    '# Stage 03 closure report — CRS04\n\n'
    f'Resultado: {stage3_status}\n\n'
    f'- Fecha UTC: {RUN_UTC}\n- Filas analíticas: {len(df)}\n'
    f'- Indicadores: {len(specs)}\n- Filas comparadas: {len(comparison)}\n'
    f'- Coincidencias estrictas SPSS–R: {strict_equal_rows}\n'
    f'- Excepciones documentadas: {documented_exception_rows}\n'
    f'- Filas validadas: {validated_rows}\n'
    f'- Filas no validadas: {len(comparison)-validated_rows}\n'
    f'- Solo SPSS: {only_spss}\n- Solo R: {only_r}\n'
    f'- Dominios válidos: {domain_valid}\n- Artefactos completos: {artifacts_complete}\n\n'
    'VS_12M Nacional se valida contra la sintaxis 3.4.1 y el denominador de 18,807; '
    'la fila consolidada SPSS original se conserva en el registro de conflictos.\n'
    'La igualdad restante usa el redondeo institucional: %/ES/IC a 1 decimal y CV a 3 decimales.\n')
(LOG_DIR/'stage3_closure_report.md').write_text(closure_text,encoding='utf-8')
(OUTPUT_DIR/'stage3_cloud_full_survey_pass.md').write_text(
    '# Stage 03 Pass — CRS04\n\n'+f'Resultado: {stage3_status}\n\n'+
    f'Fecha UTC: {RUN_UTC}\nIndicadores: {len(specs)}\n'+
    f'Coincidencias estrictas: {strict_equal_rows}/{len(comparison)}\n'+
    f'Excepciones documentadas: {documented_exception_rows}\n'+
    f'Filas validadas: {validated_rows}/{len(comparison)}\n',encoding='utf-8')
(DOCS_DIR/'stage3_description.md').write_text(
    '# Stage 03 CRS04\n\nBigQuery materializa los indicadores desde las sintaxis SPSS; '
    'R aplica el diseño complejo y reproduce exclusivamente los dominios y desagregaciones '
    'del contrato de 516 indicadores. Los microdatos permanecen fuera de GitHub.\n',encoding='utf-8')
display(coverage); display(closure)
print(stage3_status)
if len(special_pending): print('Hay dimensiones SPSS especiales pendientes; revisar stage3_spss_special_dimensions_pending.csv')
if not passed:
    raise RuntimeError(
        'Stage 3 contiene filas no validadas. Revisa stage3_spss_vs_r_differences.csv, '
        'stage3_spss_vs_r_coverage.csv y stage3_spss_reference_logic_conflicts.csv.')


## 12. Gate cloud completo y evidencia versionable


In [ ]:
EXPECTED_INDICATORS=516
EXPECTED_COMPARISON_ROWS=3014
EXPECTED_STRICT_ROWS=3013
EXPECTED_DOCUMENTED_EXCEPTIONS=1

plan_audit=pd.read_csv(LOG_DIR/'stage3_csplan_validation.csv').iloc[0]
full_gate_checks={
    'indicator_specs': len(specs)==EXPECTED_INDICATORS,
    'spss_indicators': s.indicator_id.nunique()==EXPECTED_INDICATORS,
    'comparison_rows': len(comparison)==EXPECTED_COMPARISON_ROWS,
    'strict_rows': strict_equal_rows==EXPECTED_STRICT_ROWS,
    'documented_exceptions': documented_exception_rows==EXPECTED_DOCUMENTED_EXCEPTIONS,
    'validated_rows': validated_rows==EXPECTED_COMPARISON_ROWS,
    'only_spss_rows': only_spss==0,
    'only_r_rows': only_r==0,
    'analytical_rows': int(plan_audit['filas'])==18807,
    'strata': int(plan_audit['estratos'])==25,
    'psus': int(plan_audit['conglomerados'])==1115,
    'design_df': int(plan_audit['grados_libertad'])==1090,
    'closure_passed': bool(passed),
}
failed_checks=[name for name,ok in full_gate_checks.items() if not ok]
if failed_checks:
    raise RuntimeError(f'FULL SURVEY GATE failed: {failed_checks}')

print('Indicators:',len(specs))
print('Comparison rows:',len(comparison))
print('Strict matches:',strict_equal_rows)
print('Documented exceptions:',documented_exception_rows)
print('Validated rows:',validated_rows)
print('Survey design: 18807 rows, 25 strata, 1115 PSUs, 1090 df')
print('FULL SURVEY GATE: PASS')


In [ ]:
evidence_path=OUTPUT_DIR/'stage3_cloud_full_survey_pass.md'
comparison_evidence_path=LOG_DIR/'stage3_cloud_full_survey_comparison_20260823.csv'
comparison.to_csv(comparison_evidence_path,index=False)

exception_rows=comparison.loc[comparison.EXCEPCION_DOCUMENTADA.fillna(False),
                              ['indicator_id','dimension','categoria_spss']]
exception_text='; '.join(' — '.join(map(str,row)) for row in exception_rows.to_numpy())
evidence_text=(
    '# Stage 03 cloud full survey validation — CRS04\n\n'
    'Resultado: PASS_CON_EXCEPCION_DOCUMENTADA\n\n'
    f'- Fecha UTC: {RUN_UTC}\n'
    f'- Tabla candidata: `{A}`\n'
    f'- Export SHA-256: `{sha256(export_path)}`\n'
    f'- Filas analíticas: {len(df)}\n'
    f'- Indicadores: {len(specs)}\n'
    f'- Filas comparadas: {len(comparison)}\n'
    f'- Coincidencias estrictas SPSS–R: {strict_equal_rows}\n'
    f'- Excepciones documentadas: {documented_exception_rows}\n'
    f'- Filas validadas: {validated_rows}\n'
    f'- Solo SPSS: {only_spss}\n'
    f'- Solo R: {only_r}\n'
    '- Diseño: 18,807 filas; 25 estratos; 1,115 UPM; 1,090 gl\n'
    f'- Excepción histórica: {exception_text}\n\n'
    'La referencia SPSS V0 se leyó desde `04Outputs` y no fue modificada. '
    'Todos los productos de esta ejecución se escribieron bajo `shadow_full_v0_5`.\n')
evidence_path.write_text(evidence_text,encoding='utf-8')

evidence_manifest=pd.DataFrame([
    {'file':str(evidence_path),'sha256':sha256(evidence_path)},
    {'file':str(comparison_evidence_path),'sha256':sha256(comparison_evidence_path)},
])
display(evidence_manifest)
print('Evidence:',evidence_path)
print('Comparison:',comparison_evidence_path)
print('FULL SURVEY EVIDENCE: PASS')


In [ ]:
import zipfile
bundle_path=OUTPUT_DIR/'stage3_cloud_full_survey_evidence_20260823.zip'
bundle_files=[
    evidence_path,
    comparison_evidence_path,
    LOG_DIR/'stage3_nb09_closure.csv',
    LOG_DIR/'stage3_spss_vs_r_coverage.csv',
]
with zipfile.ZipFile(bundle_path,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in bundle_files:
        bundle.write(path,arcname=path.name)
print('Evidence bundle:',bundle_path)
